# PCNSL plasma proteomics main notebook

Two low-correlation outliers (training `patient_id` 73 and 86) are
removed

## Setup


In [ ]:
import os, math, warnings, re as _re
from matplotlib.lines import Line2D
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_v] = '1'
warnings.filterwarnings('ignore')

from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Patch, Polygon, Rectangle, Wedge
from matplotlib.colors import to_rgb, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, NullLocator, FuncFormatter
from matplotlib.transforms import Bbox
import seaborn as sns
import networkx as nx
from PIL import Image as PILImage

from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial import ConvexHull

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import roc_curve

from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from statsmodels.duration.hazard_regression import PHReg
from statsmodels.duration.survfunc import SurvfuncRight, survdiff
from statsmodels.stats.multitest import multipletests

import gseapy as gp
from gseapy import Msigdb

from IPython.display import Image, SVG, display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 600,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10})

In [ ]:
#Paths
BASE_DATA      = Path('data/pcnsl/data_PCNSL_survival_olink_no_outlier.csv')
REPLICATES_CSV = Path('data/pcnsl/proteomics/data_PCNSL_survival_olink_replicates_r0955.csv')
ROOT           = Path('outputs')
PANEL_V2       = Path('outputs/figures')
for p in (ROOT, PANEL_V2):
    p.mkdir(parents=True, exist_ok=True)

#low-correlation training samples to drop before z-scoring
DROP_OUTLIERS = [73, 86]

SIGNATURE_DEFS = {
    'IFN_Chemokine':       ['CXCL9', 'CXCL10', 'IFNG', 'CCL19', 'CCL20', 'CCL3', 'CCL4'],
    'Checkpoint_Cytokine': ['IL10', 'IL6', 'PDCD1', 'PDCD1LG2', 'LAG3', 'CD86', 'TNF'],
    'Innate_Inflammatory': ['IL1B', 'IL6', 'TNF', 'CCL20', 'CCL3', 'CCL4', 'FASLG'],
    'Integrated_Immune':   ['IL10', 'IL6', 'IL1B', 'CCL19', 'PDCD1LG2', 'PDCD1', 'CXCL9',
                            'CXCL10', 'IFNG', 'LAG3', 'CD86', 'TNF', 'CCL20', 'CCL3',
                            'CCL4', 'TNFRSF13C', 'FASLG']}

In [ ]:
_METRICS: dict[str, str] = {}

def _log(label, value):
    _METRICS[str(label)] = str(value)
    print(f'{label}: {value}')

def _preview(fig_path, *, width=900):
    fig_path = str(fig_path)
    display(SVG(filename=fig_path) if fig_path.endswith('.svg') else Image(filename=fig_path, width=width))

def save_panel(fig, slug, *, also_png=False, copy=True, preview=True, out=None, close=True,
               png_dpi=300, svg_dpi=600):
    out = Path(out) if out is not None else PANEL_V2
    svg = out / f'{slug}.svg'
    fig.savefig(svg, dpi=svg_dpi, bbox_inches='tight')   # SVG -> outputs/figures_v2 only
    if preview:
        _preview(svg)
    if close:
        plt.close(fig)
    return svg

def stitch_panels(figs, layout, *, out, dpi=300, copy_to_article_name=None):
    import io
    ims = []
    for f in figs:
        _buf = io.BytesIO(); f.savefig(_buf, format='png', dpi=dpi, bbox_inches='tight'); _buf.seek(0)
        ims.append(PILImage.open(_buf).convert('RGB'))
    def _hstack(imgs):
        h = min(im.height for im in imgs)
        scaled = [im.resize((int(im.width * h / im.height), h)) for im in imgs]
        c = PILImage.new('RGB', (sum(im.width for im in scaled), max(im.height for im in scaled)), 'white')
        x = 0
        for im in scaled:
            c.paste(im, (x, 0)); x += im.width
        return c
    def _vstack(imgs):
        w = min(im.width for im in imgs)
        scaled = [im.resize((w, int(im.height * w / im.width))) for im in imgs]
        c = PILImage.new('RGB', (max(im.width for im in scaled), sum(im.height for im in scaled)), 'white')
        y = 0
        for im in scaled:
            c.paste(im, (0, y)); y += im.height
        return c
    combo = {'horizontal': _hstack, '1x4': _hstack, 'vertical': _vstack, '5x1': _vstack,
             '2x2': lambda im: _vstack([_hstack(im[:2]), _hstack(im[2:])]),
             '2x3': lambda im: _vstack([_hstack(im[:3]), _hstack(im[3:])])}[layout](ims)
    out = Path(out); out.parent.mkdir(parents=True, exist_ok=True)
    combo.save(str(out), 'PDF', resolution=float(dpi))
    if copy_to_article_name:
        copy_to_article(out, copy_to_article_name)
    return out

def mskcc_like(age, kps):
    if pd.isna(age) or pd.isna(kps): return np.nan
    if age < 50: return 1
    if kps >= 70: return 2
    return 3

def _stars(fdr):
    if pd.isna(fdr): return ''
    if fdr < 0.001: return '***'
    if fdr < 0.01: return '**'
    if fdr < 0.05: return '*'
    return ''

ENDPOINTS = [
    ('OS',  'estimated_OS_months',  'estimated_OS_event'),
    ('PFS', 'estimated_PFS_months', 'estimated_PFS_event'),
]

def logrank_p(time, event, group):
    try:
        _, p = survdiff(np.asarray(time, float), np.asarray(event, int), np.asarray(group, int))
        return float(p)
    except Exception:
        return float('nan')

def km_step(times, events):
    sf = SurvfuncRight(np.asarray(times, float), np.asarray(events, int))
    return np.r_[0.0, sf.surv_times], np.r_[1.0, sf.surv_prob]

def km_at(times, events, t_query):
    sf = SurvfuncRight(np.asarray(times, float), np.asarray(events, int))
    if len(sf.surv_times) == 0:
        return 1.0, 1.0, 1.0
    i = max(0, min(np.searchsorted(sf.surv_times, float(t_query), side='right') - 1, len(sf.surv_prob) - 1))
    s = float(sf.surv_prob[i])
    se = float(sf.surv_prob_se[i]) if len(sf.surv_prob_se) > i else 0.0
    return s, max(0.0, s - 1.96 * se), min(1.0, s + 1.96 * se)

def add_risk_table(rax, sub, group_col, time_col, *, times,
                   group_order=('Low', 'High'), row_labels=None,
                   show_header=True, show_row_labels=True, fontsize=9):
    rax.axis('off')
    row_labels = row_labels or list(group_order)

    xs = np.linspace(0.24, 0.95, len(times))
    title_y = 0.92
    header_y = 0.66
    row_ys = [0.38, 0.12] if len(group_order) == 2 else list(np.linspace(0.42, 0.10, len(group_order)))

    if show_header:
        rax.text(0.0, title_y, 'Patients at risk',
                 fontweight='bold', fontsize=fontsize, transform=rax.transAxes)
        for x, t in zip(xs, times):
            rax.text(x, header_y, str(int(t)), ha='center',
                     fontsize=fontsize, transform=rax.transAxes)

    if show_row_labels:
        for y, lab in zip(row_ys, row_labels):
            rax.text(0.0, y, lab, fontsize=fontsize, transform=rax.transAxes)

    t = pd.to_numeric(sub[time_col], errors='coerce').values
    g = sub[group_col].astype(str).values

    for y, grp in zip(row_ys, group_order):
        m = g == grp
        for x, tt in zip(xs, times):
            rax.text(x, y, str(int(((t >= tt) & m).sum())),
                     ha='center', fontsize=fontsize, transform=rax.transAxes)

def plot_km(ax, sub, time_col, event_col, group_col, *,
            palette=None, group_order=None, xmax=None,
            xlabel='Months', ylabel='Survival probability', title=None,
            censor_ticks=True, p_box=True, p_groups=None, lw=2.0,
            legend_loc='upper right', legend_fontsize=8, p_text=None,
            p_style='italic', p_loc=(0.03, 0.08), p_ha='left'):
    t_all = pd.to_numeric(sub[time_col], errors='coerce')
    e_all = pd.to_numeric(sub[event_col], errors='coerce')
    groups = sub[group_col].astype(str).values
    group_order = group_order or list(pd.unique(groups))
    if palette is None:
        palette = {g: c for g, c in zip(group_order, ['#c44e52', '#4c78a8', '#2ca02c', '#ff7f0e'])}
    for grp in group_order:
        m = groups == grp
        if not m.any(): continue
        ts = t_all[m].values; es = e_all[m].values
        ok = np.isfinite(ts) & np.isfinite(es)
        ts, es = ts[ok], es[ok].astype(int)
        if len(ts) == 0: continue
        xs, ys = km_step(ts, es)
        col = palette.get(grp, '#888')
        ax.step(xs, ys, where='post', lw=lw, color=col, label=grp)
        if censor_ticks:
            for ct in ts[es == 0]:
                i = max(0, min(np.searchsorted(xs, ct, side='right') - 1, len(ys) - 1))
                ax.plot([ct], [ys[i]], marker='|', color=col, ms=6, mew=1.1)
    if p_box:
        if p_groups is None and len(group_order) == 2:
            p_groups = tuple(group_order)
        if p_groups is not None:
            m = np.isin(groups, p_groups)
            p = logrank_p(t_all[m], e_all[m], (groups[m] == p_groups[0]).astype(int))
            txt = p_text(p) if p_text else ('P = NA' if not np.isfinite(p) else ('P < 0.001' if p < 0.001 else f'P = {p:.3g}'))
            kw = dict(transform=ax.transAxes, ha=p_ha, va='bottom', fontsize=8 if p_style == 'box' else 10.2)
            if p_style == 'italic': kw['style'] = 'italic'
            if p_style == 'box':
                kw['bbox'] = dict(boxstyle='round,pad=0.25', fc='white', ec='lightgray', alpha=0.9)
            ax.text(p_loc[0], p_loc[1], txt, **kw)
    if xmax is None:
        finite = t_all.dropna().values
        xmax = float(max(finite.max() if len(finite) else 24.0, 24.0))
    ax.set_xlim(0, xmax); ax.set_ylim(0, 1.03)
    if xlabel is not None: ax.set_xlabel(xlabel)
    if ylabel is not None: ax.set_ylabel(ylabel)
    if title: ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.22)
    if legend_loc:
        ax.legend(frameon=False, fontsize=legend_fontsize, loc=legend_loc)

def km_risk_pair(fig, outer_gs, row, col, sub, tcol, ecol, gcol, *, times, xmax, title,
                 palette, group_order=('High', 'Low'), risk_order=('Low', 'High')):
    inner = outer_gs[row, col].subgridspec(2, 1, height_ratios=[3.0, 0.85], hspace=0.28)
    plot_km(fig.add_subplot(inner[0, 0]), sub, tcol, ecol, gcol,
            palette=palette, group_order=group_order, xmax=xmax, title=title,
            legend_loc='upper right', lw=2.2, p_groups=tuple(group_order))
    add_risk_table(fig.add_subplot(inner[1, 0]), sub, gcol, tcol,
                   times=times, group_order=risk_order)

def fit_cox_hr(T, E, X, var_names=None):
    T = np.asarray(T, float); E = np.asarray(E, int); X = np.asarray(X, float)
    if X.ndim == 1: X = X[:, None]
    p = X.shape[1]
    try:
        m = PHReg(T, X, status=E).fit(disp=0)
        ci = m.conf_int()
        return {'hr': np.exp(m.params), 'lo': np.exp(ci[:, 0]), 'hi': np.exp(ci[:, 1]),
                'p': m.pvalues, 'n': len(T), 'events': int(E.sum()),
                'var_names': list(var_names) if var_names else None, 'model': m}
    except Exception:
        nan = np.full(p, np.nan)
        return {'hr': nan, 'lo': nan, 'hi': nan, 'p': nan, 'n': len(T),
                'events': int(E.sum()), 'var_names': var_names, 'model': None}

def draw_hr_point(ax, hr, lo, hi, y, *, color, filled=True, star='', xtext=1.04):
    if not np.isfinite(hr): return
    ax.plot([lo, hi], [y, y], color=color, lw=1.8, alpha=0.85 if not filled else 1.0, zorder=2)
    if filled:
        ax.scatter([hr], [y], color=color, s=50, edgecolor='black', linewidth=0.35, zorder=3)
    else:
        ax.scatter([hr], [y], facecolor='white', edgecolor=color, s=50, linewidth=1.5, zorder=3)
    if star:
        ax.text(max(hi, 1.05) * xtext, y, star, va='center', fontsize=8,
                fontweight='bold', color=color, style='normal' if filled else 'italic')

def draw_pathway_forest(ax, rows_u, rows_a, y_labels, *, color, y_offset=0.12):
    y = np.arange(len(y_labels))[::-1]
    ax.axvline(1, color='black', ls='--', lw=1.0, zorder=1)
    for i, r in enumerate(rows_u):
        draw_hr_point(ax, r['HR'], r['HR_lo'], r['HR_hi'], y[i] + y_offset, color=color,
                      filled=True, star=_stars(r.get('fdr')))
    for i, r in enumerate(rows_a):
        draw_hr_point(ax, r['HR'], r['HR_lo'], r['HR_hi'], y[i] - y_offset, color=color,
                      filled=False, star=_stars(r.get('fdr')))
    ax.set_yticks(y); ax.set_yticklabels(y_labels, fontsize=8)
    return y

def plot_gsea_bar(res_df, *, slug, cohort_label, display_name, top_n=15):
    viz = res_df.sort_values('NES', key=lambda s: s.abs(), ascending=False).head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(5, 5))
    colors = ['#c44e52' if x > 0 else '#4c78a8' for x in viz['NES']]
    ax.barh(range(len(viz)), viz['NES'], color=colors, edgecolor='black', lw=0.5)
    ax.set_yticks(range(len(viz)))
    labels = [t if len(t) <= 55 else t[:52] + '...' for t in viz['pretty']]
    ax.set_yticklabels(labels, fontsize=7.5 if slug == 'c7' else 8.5)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel('Normalised Enrichment Score (NES)')
    coh = 'Training' if cohort_label == 'train' else 'Test'
    ax.set_title(f'{coh} cohort — {display_name} pre-ranked GSEA', fontsize=10)
    pad = 0.06 * (ax.get_xlim()[1] - ax.get_xlim()[0])
    for i, (_, row) in enumerate(viz.iterrows()):
        sig = _stars(row['FDR q-val'])
        if sig:
            nes = row['NES']
            ax.text(nes + (pad if nes >= 0 else -pad), i, sig, va='center',
                    ha='left' if nes >= 0 else 'right', fontsize=10, color='#333', fontweight='bold')
    if cohort_label == 'train':
        ax.legend([Patch(color='#c44e52'), Patch(color='#4c78a8')],
                  ['Up in Inflamed', 'Up in Non-inflamed'], loc='upper center',
                  bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False, fontsize=8)
    fig.tight_layout()
    out = PANEL_V2 / f'supp_fig_s4_gsea_{slug}_{cohort_label}.svg'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    return out

def draw_sig_network(ax, G, pos, gene_mem, sig_colors, *, n_cohort, n_drawn, label):
    for u, v, d in G.edges(data=True):
        rho = d.get('rho', d.get('weight'))
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                color='#2166ac' if rho > 0 else '#b2182b',
                lw=0.5 + 4.0 * (abs(rho) - 0.30) / 0.70, alpha=0.6, zorder=1)
    r = 0.06
    for nd in G.nodes():
        cx, cy = pos[nd]
        cols = [sig_colors[m] for m in gene_mem[nd]] if gene_mem[nd] else ['#bbbbbb']
        for i, c in enumerate(cols):
            ax.add_patch(Wedge((cx, cy), r, i * 360 / len(cols), (i + 1) * 360 / len(cols),
                               facecolor=c, edgecolor='none', zorder=3))
        ax.add_patch(Wedge((cx, cy), r, 0, 360, facecolor='none', edgecolor='black', lw=0.8, zorder=4))
        ax.text(cx, cy + r * 1.4, nd, fontsize=9, ha='center', va='bottom', zorder=5)
    ax.set_aspect('equal'); ax.set_axis_off()
    xs, ys = [p[0] for p in pos.values()], [p[1] for p in pos.values()]
    m = r * 3
    ax.set_xlim(min(xs) - m, max(xs) + m); ax.set_ylim(min(ys) - m, max(ys) + m)
    ax.set_title(f'{label.title()} cohort (n={n_cohort}) — {n_drawn} edges', fontsize=10)


## Data preparation

Load the proteomics CSV, drop the outlier training samples, compute the four pre-specified
immune signature scores


In [ ]:
df_raw = pd.read_csv(BASE_DATA)
_log('Input CSV rows', df_raw.shape[0])
_log('Input CSV columns', df_raw.shape[1])

_pid = df_raw['patient_id'].astype(str)
_replicate_mask = _pid.str.contains('/')
_outlier_mask   = _pid.isin([str(x) for x in DROP_OUTLIERS]) & (df_raw['cohort'] == 'training')
_drop_mask = _replicate_mask | _outlier_mask
df = df_raw.loc[~_drop_mask].reset_index(drop=True).copy()
_log('Replicate rows dropped', int(_replicate_mask.sum()))
_log('Outlier rows dropped',   list(df_raw.loc[_outlier_mask, 'patient_id'].astype(str)))

meta_cols = {
    'patient_id', 'sex_completed', 'age_years', 'cohort',
    'Karnofsky_status', 'Karnofsky_group',
    'estimated_OS_months', 'estimated_OS_event',
    'estimated_PFS_months', 'estimated_PFS_event'}

protein_cols_all = [c for c in df.columns if c not in meta_cols]

Outlier dropping reasoning

In [ ]:
_qc_prot = [c for c in df_raw.columns if c not in meta_cols]                       
_qc = df_raw.loc[~df_raw['patient_id'].astype(str).str.contains('/')].copy()       
_Xqc = _qc[_qc_prot].apply(pd.to_numeric, errors='coerce')
_Xqc = _Xqc.fillna(_Xqc.median(axis=0))                                            
_corr = np.corrcoef(_Xqc.to_numpy(float))                                          
np.fill_diagonal(_corr, np.nan)                                                    
_mean_corr = pd.Series(np.nanmean(_corr, axis=1), index=_qc['patient_id'].astype(str).values)
for _pid in DROP_OUTLIERS:                                                         
    _log(f'Mean sample-to-sample correlation (patient {_pid})', round(float(_mean_corr.get(str(_pid), np.nan)), 3))
_log('Cohort median mean-correlation', round(float(_mean_corr.median()), 3))

In [ ]:
# NOT USED
for cohort_name, idx in df.groupby(df['cohort'].astype(str).str.lower()).groups.items():
    sub = df.loc[idx, protein_cols_all].apply(pd.to_numeric, errors='coerce')
    mu = sub.mean(axis=0)
    sd = sub.std(axis=0, ddof=0).replace(0, np.nan)
    z = ((sub - mu) / sd).fillna(0.0)
    for c in protein_cols_all:
        df.loc[idx, f'z_{c}'] = z[c].values

for sig, markers in SIGNATURE_DEFS.items():
    cols = [m for m in markers if m in df.columns]
    if cols:
        df[sig] = df[cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# Immune-Apoptotic 7-protein composite (CXCL13 absent, so 5/6 adverse proteins used)
APOPT_ADVERSE    = ['ABL1', 'GDNF', 'EDA2R', 'IL1B', 'CASP1']
APOPT_PROTECTIVE = 'TNFRSF10B'
_apopt_cols = [p for p in APOPT_ADVERSE if p in df.columns]
df['Immune_Apoptotic'] = (
    df[_apopt_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    - pd.to_numeric(df[APOPT_PROTECTIVE], errors='coerce'))

# derived clinical covariates
_age = pd.to_numeric(df['age_years'],        errors='coerce')
_kps = pd.to_numeric(df['Karnofsky_status'], errors='coerce')
df['male']      = (df['sex_completed'].astype(str).str.upper().str[0] == 'M').astype(int)
df['age_dec']   = (_age - 60) / 10
df['age_ge60']  = (_age >= 60).astype(int)
df['kps_ge70']  = (_kps >= 70).astype(int)
df['MSKCC_like'] = [mskcc_like(a, k) for a, k in zip(_age, _kps)]
df['bad_responder'] = ((pd.to_numeric(df['estimated_PFS_event'], errors='coerce') == 1) &
                       (pd.to_numeric(df['estimated_PFS_months'], errors='coerce') < 9)).astype(int)

_os18 = pd.to_numeric(df['estimated_OS_months'], errors='coerce')
_ev18 = pd.to_numeric(df['estimated_OS_event'],  errors='coerce')
df['long_survivor'] = np.where(_os18 > 18, 1.0,
                       np.where((_os18 <= 18) & (_ev18 == 1), 0.0, np.nan))

# Cohort masks
is_train = df['cohort'].astype(str).str.lower().eq('training')
is_test  = df['cohort'].astype(str).str.lower().eq('validation')

# High/Low groups
for _sig in SIGNATURE_DEFS:
    _thr = df.loc[is_train, _sig].median()
    df[f'{_sig}_group'] = np.where(pd.to_numeric(df[_sig], errors='coerce') >= _thr,
                                    'High', 'Low')
for _cohort, _mask in df.groupby(df['cohort'].astype(str).str.lower()).groups.items():
    _thr = df.loc[_mask, 'Immune_Apoptotic'].median()
    df.loc[_mask, 'Immune_Apoptotic_group'] = np.where(
        pd.to_numeric(df.loc[_mask, 'Immune_Apoptotic'], errors='coerce') >= _thr, 'High', 'Low')

AUGMENTED_CSV = ROOT / 'data_PCNSL_survival_olink_augmented_for_notebook.csv'
df.to_csv(AUGMENTED_CSV, index=False)

n_train = int(is_train.sum())
n_test  = int(is_test.sum())
n_prot  = len(protein_cols_all)
_log('n training (after outlier drop)',    n_train)
_log('n test',                             n_test)
_log('n unique proteins',                  n_prot)
_log('Immune_Apoptotic adverse proteins',  APOPT_ADVERSE)
_log('Immune_Apoptotic protective protein', APOPT_PROTECTIVE)

display(df[['patient_id', 'cohort', 'age_years', 'Karnofsky_status',
            'estimated_OS_months', 'estimated_PFS_months',
            'IFN_Chemokine', 'Checkpoint_Cytokine',
            'Innate_Inflammatory', 'Integrated_Immune',
            'Immune_Apoptotic', 'Immune_Apoptotic_group']].head())

### Baseline cohort characteristics (Table 1)


In [ ]:
df_full = df[is_train | is_test].copy()

for mask_, coh_label in [(is_train, 'Training'), (is_test, 'Test')]:
    sub = df.loc[mask_]
    a   = pd.to_numeric(sub['age_years'], errors='coerce')
    k   = pd.to_numeric(sub['Karnofsky_status'], errors='coerce')
    os_m  = pd.to_numeric(sub['estimated_OS_months'],  errors='coerce')
    pfs_m = pd.to_numeric(sub['estimated_PFS_months'], errors='coerce')
    os_e  = pd.to_numeric(sub['estimated_OS_event'],   errors='coerce').fillna(0).astype(int)
    pfs_e = pd.to_numeric(sub['estimated_PFS_event'],  errors='coerce').fillna(0).astype(int)

    _log(f'Table 1 {coh_label} n', len(sub))
    _log(f'Table 1 {coh_label} median age (IQR)',
         f'{a.median():.0f} ({a.quantile(0.25):.0f}-{a.quantile(0.75):.0f})')
    _log(f'Table 1 {coh_label} male n (%)',
         f"{int(sub['male'].sum())} ({100*sub['male'].mean():.0f}%)")
    _log(f'Table 1 {coh_label} KPS>=70 n (%)',
         f"{int(sub['kps_ge70'].sum())} ({100*sub['kps_ge70'].mean():.0f}%)")

    msk = sub['MSKCC_like']
    for cls in (1, 2, 3):
        n = int((msk == cls).sum())
        _log(f'Table 1 {coh_label} MSKCC class {cls} n (%)',
             f'{n} ({100*n/len(msk):.0f}%)')

    _log(f'Table 1 {coh_label} OS events n (%)',
         f'{int(os_e.sum())} ({100*os_e.mean():.0f}%)')
    _log(f'Table 1 {coh_label} PFS events n (%)',
         f'{int(pfs_e.sum())} ({100*pfs_e.mean():.0f}%)')
    _log(f'Table 1 {coh_label} median OS months (IQR)',
         f'{os_m.median():.1f} ({os_m.quantile(0.25):.1f}-{os_m.quantile(0.75):.1f})')
    _log(f'Table 1 {coh_label} median PFS months (IQR)',
         f'{pfs_m.median():.1f} ({pfs_m.quantile(0.25):.1f}-{pfs_m.quantile(0.75):.1f})')

_os_all  = pd.to_numeric(df_full['estimated_OS_months'],  errors='coerce')
_pfs_all = pd.to_numeric(df_full['estimated_PFS_months'], errors='coerce')
_log('Overall median OS months',  f'{_os_all.median():.1f}')
_log('Overall median PFS months', f'{_pfs_all.median():.1f}')

## 1 - Clustering and signature selection

k-means clustering, signature matching of clusters, definition of heatmap proteins, signature high/low split analysis

### Feature matrix and consensus clustering setup


In [ ]:
#consensus k-means on training, project to test, then KM/Cox
dc = df.copy()
exclude = {
    'patient_id','Provenance','Protocole','sexe','sex_completed','sex_source','age_years','cohort',
    'Karnofsky_status','Karnofsky_group','estimated_overall_risk_score','estimated_overall_risk_group',
    'estimated_OS_months','estimated_OS_event','estimated_PFS_months','estimated_PFS_event',
    
}
sig_roots = ['IFN_Chemokine','Checkpoint_Cytokine','Innate_Inflammatory','Integrated_Immune']
_derived_meta = {'male', 'age_dec', 'age_ge60', 'kps_ge70', 'MSKCC_like',
                 'bad_responder', 'long_survivor', 'outlier_score'}
protein_cols = [
    c for c in dc.columns
    if c not in exclude and c not in _derived_meta and not c.startswith('z_') and c not in sig_roots and not c.endswith('_group') and not any(c.startswith(r+'_') for r in sig_roots)
]
immune_sigs = {
    'IFN/Chemokine':['CXCL9','CXCL10','IFNG','CCL19','CCL20','CCL3','CCL4'],
    'Checkpoint/Cytokine':['IL10','IL6','PDCD1','PDCD1LG2','LAG3','CD86','TNF'],
    'Innate/Inflammatory':['IL1B','IL6','TNF','CCL20','CCL3','CCL4','FASLG'],
    'Integrated immune':['IL10','IL6','IL1B','CCL19','PDCD1LG2','PDCD1','CXCL9','CXCL10','IFNG','LAG3','CD86','TNF','CCL20','CCL3','CCL4','TNFRSF13C','FASLG']}

for name, genes in immune_sigs.items():
    keep = [g for g in genes if g in dc.columns]
    dc[name + '_score'] = dc[keep].apply(pd.to_numeric, errors='coerce').mean(axis=1)
cohort = dc['cohort'].astype(str).str.lower()
train_mask = cohort.eq('training').values
test_mask = cohort.eq('validation').values
train_df = dc.loc[train_mask].copy().reset_index(drop=True)
test_df = dc.loc[test_mask].copy().reset_index(drop=True)
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[protein_cols].to_numpy(float))
X_test = scaler.transform(test_df[protein_cols].to_numpy(float))

X_train = X_train - np.median(X_train, axis=1, keepdims=True)
X_test = X_test - np.median(X_test, axis=1, keepdims=True)

### consensus bootstrap k-means and k selection


In [ ]:
BOOTSTRAPS = 500
RESAMPLE_FRAC = 0.8

rng = np.random.default_rng(42)

metrics_rows = []
consensus_store = {}
model_store = {}

for k in range(2, 6):
    n = len(train_df)
    m = int(math.ceil(RESAMPLE_FRAC * n))
    together = np.zeros((n, n), dtype=np.uint16)
    same = np.zeros((n, n), dtype=np.uint16)
    for b in range(BOOTSTRAPS):
        idx = np.sort(rng.choice(n, size=m, replace=False))
        labs = KMeans(n_clusters=k, n_init=1, random_state=1000+b, max_iter=80, algorithm='lloyd').fit_predict(X_train[idx])
        together[np.ix_(idx, idx)] += 1
        for lab in np.unique(labs):
            mem = idx[labs == lab]
            same[np.ix_(mem, mem)] += 1
    
    cons = np.divide(same, together, out=np.zeros_like(same, dtype=float), where=together > 0)
    final_model = KMeans(n_clusters=k, n_init=100, random_state=42, max_iter=300, algorithm='lloyd').fit(X_train)
    labels = final_model.labels_
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    upper = cons[np.triu_indices(n, 1)]
    pac = ((upper > 0.1) & (upper < 0.9)).mean()
    sil = silhouette_score(X_train, labels)
    metrics_rows.append({'k': k, 'PAC': float(pac), 'silhouette': float(sil), 'cluster_sizes': ';'.join(map(str, sizes)), 'min_cluster_size': int(min(sizes)), 'max_cluster_size': int(max(sizes))})
    consensus_store[k] = cons
    model_store[k] = final_model

metrics_df = pd.DataFrame(metrics_rows)
eligible = metrics_df.query('min_cluster_size >= 8').copy()
if eligible.empty: eligible = metrics_df.copy()
eligible = eligible.sort_values(['PAC', 'silhouette', 'k'], ascending=[True, False, True]).reset_index(drop=True)

best_k = int(eligible.loc[0, 'k'])
k2 = metrics_df.loc[metrics_df['k'] == 2].iloc[0]
if k2['min_cluster_size'] >= 15 and k2['PAC'] <= eligible.loc[0, 'PAC'] + 0.10: best_k = 2

### final fit, cluster label assignment, train/test projection

In [ ]:
final_model = model_store[best_k]

train_labels = final_model.labels_
test_labels = final_model.predict(X_test)
train_cluster_means = train_df.groupby(train_labels)['Integrated immune_score'].mean().sort_values(ascending=False)
inflamed_cluster = int(train_cluster_means.index[0])

full_labels = np.empty(len(dc), dtype=int)
full_labels[train_mask] = train_labels
full_labels[test_mask] = test_labels

dc['kmeans_cluster'] = full_labels + 1
dc['serum_group'] = np.where(full_labels == inflamed_cluster, 'Inflamed serum', 'Non-inflamed serum')
dc['cohort_plot'] = np.where(train_mask, 'Training', 'Test')

train_df = dc.loc[train_mask].copy().reset_index(drop=True)
test_df = dc.loc[test_mask].copy().reset_index(drop=True)

### clinical correlates of the inflammatory serum cluster

Tests whether baseline clinical characteristics (age, sex, KPS, MSKCC-like class, cohort) differ between the Inflamed and Non-inflamed serum groups. A lack of association indicates the inflammatory profile is not merely a proxy for established clinical risk factors. Reviewer request (C. Houillier).


In [ ]:
clin_corr_dir = ROOT / 'clinical_correlates'
clin_corr_dir.mkdir(parents=True, exist_ok=True)

_cc = dc.copy()
_cc['_infl'] = _cc['serum_group'].eq('Inflamed serum')
_cc_age = pd.to_numeric(_cc['age_years'], errors='coerce')
_cc_kps = pd.to_numeric(_cc['Karnofsky_status'], errors='coerce')


def _cc_fisher(name, binary_series):
    b = pd.to_numeric(binary_series, errors='coerce')
    m = b.notna()
    g = _cc.loc[m, '_infl'].values
    b = b[m].astype(int).values
    a11 = int(((b == 1) & g).sum()); a10 = int(((b == 1) & ~g).sum())
    a01 = int(((b == 0) & g).sum()); a00 = int(((b == 0) & ~g).sum())
    _, p = stats.fisher_exact([[a11, a10], [a01, a00]])
    it, nt = a11 + a01, a10 + a00
    return {'variable': name, 'test': 'Fisher exact',
            'Inflamed': f'{a11}/{it} ({100*a11/it:.0f}%)' if it else 'NA',
            'Non-inflamed': f'{a10}/{nt} ({100*a10/nt:.0f}%)' if nt else 'NA',
            'p_value': float(p)}


def _cc_mwu(name, cont_series):
    x = pd.to_numeric(cont_series, errors='coerce')
    xi = x[_cc['_infl'] & x.notna()]
    xn = x[~_cc['_infl'] & x.notna()]
    p = (stats.mannwhitneyu(xi, xn, alternative='two-sided').pvalue
         if len(xi) > 1 and len(xn) > 1 else np.nan)
    return {'variable': name, 'test': 'Mann-Whitney U',
            'Inflamed': f'{xi.median():.0f} [{xi.quantile(.25):.0f}-{xi.quantile(.75):.0f}]',
            'Non-inflamed': f'{xn.median():.0f} [{xn.quantile(.25):.0f}-{xn.quantile(.75):.0f}]',
            'p_value': float(p)}


def _cc_chi2(name, cat_series):
    m = cat_series.notna()
    tab = pd.crosstab(cat_series[m], _cc.loc[m, '_infl'])
    chi2, p, _, _ = stats.chi2_contingency(tab)
    infl = '/'.join(str(int(tab[True].get(idx, 0))) if True in tab else '0' for idx in tab.index)
    non = '/'.join(str(int(tab[False].get(idx, 0))) if False in tab else '0' for idx in tab.index)
    return {'variable': f'{name} (classes {list(tab.index)})', 'test': 'Chi-square',
            'Inflamed': infl, 'Non-inflamed': non, 'p_value': float(p)}


clin_corr = pd.DataFrame([
    _cc_mwu('Age, years (median [IQR])', _cc_age),
    _cc_fisher('Age >= 60', (_cc_age >= 60).astype('Int64')),
    _cc_fisher('Male sex', _cc['male']),
    _cc_fisher('KPS >= 70', (_cc_kps >= 70).astype('Int64')),
    _cc_chi2('MSKCC-like class', _cc['MSKCC_like']),
    _cc_fisher('Test cohort', _cc['cohort_plot'].eq('Test').astype(int)),
])
_ccm = clin_corr['p_value'].notna()
clin_corr.loc[_ccm, 'BH_FDR'] = multipletests(clin_corr.loc[_ccm, 'p_value'], method='fdr_bh')[1]
clin_corr.to_csv(clin_corr_dir / 'cluster_clinical_correlates.csv', index=False)

_cc_sig = ', '.join(clin_corr.loc[clin_corr['BH_FDR'] < 0.05, 'variable']) or 'none'
_log('Clinical variables associated with serum cluster (BH-FDR<0.05)', _cc_sig)
display(clin_corr)


### per-protein differential abundance (Welch's t, BH FDR)


In [ ]:
rows = []
for prot in protein_cols:
    out = {'protein': prot}
    for prefix, subdf in [('train', train_df), ('test', test_df)]:
        mask = subdf['serum_group'].eq('Inflamed serum')
        x1 = subdf.loc[mask, prot].astype(float).values
        x0 = subdf.loc[~mask, prot].astype(float).values
        _, p = stats.ttest_ind(x1, x0, equal_var=False, nan_policy='omit')
        if not np.isfinite(p): p = 1.0
        out[f'{prefix}_log2FC'] = float(np.nanmean(x1) - np.nanmean(x0))
        out[f'{prefix}_p_value'] = float(p)
    rows.append(out)

de = pd.DataFrame(rows)

for prefix in ['train', 'test']:
    de[f'{prefix}_adj_p_value'] = multipletests(de[f'{prefix}_p_value'], method='fdr_bh')[1]
    de[f'{prefix}_significant'] = (de[f'{prefix}_adj_p_value'] < 0.05) & (de[f'{prefix}_log2FC'].abs() > 1)

de['direction_concordant'] = np.sign(de['train_log2FC']) == np.sign(de['test_log2FC'])
de['significant_both'] = de['train_significant'] & de['test_significant'] & de['direction_concordant']
de['train_neglog10_adj_p'] = -np.log10(np.clip(de['train_adj_p_value'], 1e-300, 1))
de['test_neglog10_adj_p'] = -np.log10(np.clip(de['test_adj_p_value'], 1e-300, 1))
de = de.sort_values(['train_significant', 'train_adj_p_value', 'train_log2FC'], ascending=[False, True, False]).reset_index(drop=True)

### hypergeometric immune-signature enrichment + signature selection


In [ ]:
immune_term_names = list(immune_sigs.keys())
enr_rows = []

M = len(protein_cols)

for prefix in ['train', 'test']:
    up = set(de.loc[(de[f'{prefix}_significant']) & (de[f'{prefix}_log2FC'] > 0), 'protein'])
    N = len(up)
    for term in immune_term_names:
        genes = immune_sigs[term]
        gset = set([g for g in genes if g in protein_cols])
        nset = len(gset)
        x = len(up & gset)
        p = stats.hypergeom.sf(x-1, M, nset, N) if (nset and N) else 1.0
        enr_rows.append({'cohort': prefix.title(), 'term': term, 'overlap': x, 'set_size': nset, 'gene_ratio': x/nset if nset else 0.0, 'p_value': p})

enr = pd.DataFrame(enr_rows)
enr['adj_p_value'] = enr.groupby('cohort')['p_value'].transform(lambda s: multipletests(s, method='fdr_bh')[1])
ranked = enr.loc[enr['cohort'].eq('Train')].copy().sort_values(['adj_p_value', 'p_value', 'gene_ratio', 'overlap', 'term'], ascending=[True, True, False, False, True]).reset_index(drop=True)

selected_signature = ranked.loc[0, 'term']
selected_score_col = selected_signature + '_score'
selected_threshold = float(train_df[selected_score_col].median())

dc['selected_signature_name'] = selected_signature
dc['selected_signature_score'] = dc[selected_score_col]
dc['selected_signature_group'] = np.where(dc[selected_score_col] >= selected_threshold, f'High {selected_signature}', f'Low {selected_signature}')
dc['selected_signature_binary'] = dc['selected_signature_group'].str.startswith('High').astype(int)

train_df = dc.loc[train_mask].copy().reset_index(drop=True)
test_df = dc.loc[test_mask].copy().reset_index(drop=True)
label_map = {'age_ge_60': 'Age ≥ 60', 'male': 'Male sex', 'kps_ge_70': 'KPS ≥ 70', 'sig_high': f'High {selected_signature}'}

### multivariable Cox fits (selected signature)


In [ ]:
def fit_cox(subdf, time_col, event_col, endpoint, cohort_name):
    tmp = pd.DataFrame({'time': subdf[time_col].astype(float).values, 'event': subdf[event_col].astype(int).values, 'age_ge_60': (subdf['age_years'] >= 60).astype(int).values, 'male': subdf['sex_completed'].astype(str).str.upper().eq('M').astype(int).values, 'kps_ge_70': (subdf['Karnofsky_status'] >= 70).astype(int).values, 'sig_high': subdf['selected_signature_binary'].astype(int).values})
    try:
        res = PHReg.from_formula('time ~ age_ge_60 + male + kps_ge_70 + sig_high', status=tmp['event'], data=tmp).fit(disp=0)
        ci = res.conf_int()
        out = pd.DataFrame({'variable': res.model.exog_names, 'coef': res.params, 'ci_low_coef': ci[:, 0], 'ci_high_coef': ci[:, 1], 'p_value': res.pvalues})
        out['HR'] = np.exp(out['coef'])
        out['HR_low'] = np.exp(out['ci_low_coef'])
        out['HR_high'] = np.exp(out['ci_high_coef'])
    except Exception:
        out = pd.DataFrame({'variable': ['age_ge_60', 'male', 'kps_ge_70', 'sig_high'], 'coef': np.nan, 'ci_low_coef': np.nan, 'ci_high_coef': np.nan, 'p_value': np.nan, 'HR': np.nan, 'HR_low': np.nan, 'HR_high': np.nan})
    out['endpoint'] = endpoint
    out['cohort'] = cohort_name
    return out
cox_all = pd.concat([fit_cox(train_df, 'estimated_OS_months', 'estimated_OS_event', 'OS', 'Training'), fit_cox(test_df, 'estimated_OS_months', 'estimated_OS_event', 'OS', 'Test'), fit_cox(train_df, 'estimated_PFS_months', 'estimated_PFS_event', 'PFS', 'Training'), fit_cox(test_df, 'estimated_PFS_months', 'estimated_PFS_event', 'PFS', 'Test')], ignore_index=True)

### helper plot functions


In [ ]:
palette = {'Inflamed serum': '#c44e52', 'Non-inflamed serum': '#4c78a8'}
sig_palette = {f'High {selected_signature}': '#c44e52', f'Low {selected_signature}': '#4c78a8'}
pca = PCA(n_components=2, random_state=42)
coords_train = pca.fit_transform(X_train)
coords_test = pca.transform(X_test)
var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
pos = de.loc[(de['train_significant']) & (de['train_log2FC'] > 0)].sort_values(['train_adj_p_value', 'train_log2FC'], ascending=[True, False]).head(12)['protein'].tolist()
neg = de.loc[(de['train_significant']) & (de['train_log2FC'] < 0)].sort_values(['train_adj_p_value', 'train_log2FC'], ascending=[True, True]).head(12)['protein'].tolist()
sel = pos + neg
if len(sel) < 20: sel = de.head(24)['protein'].tolist()
heat = dc[sel].copy().T
heat = heat.sub(heat.mean(axis=1), axis=0).div(heat.std(axis=1).replace(0, 1), axis=0)
train_global = np.where(train_mask)[0]
test_global = np.where(test_mask)[0]
global_order = []

for cohort_name, cdf2, idxmap in [('Training', train_df, train_global), ('Test', test_df, test_global)]:
    for grp in ['Inflamed serum', 'Non-inflamed serum']:
        tmp = cdf2.loc[cdf2['serum_group'].eq(grp)].copy()
        tmp['_local_index'] = tmp.index
        tmp = tmp.sort_values('Integrated immune_score', ascending=False)
        global_order.extend(idxmap[tmp['_local_index'].to_numpy()].tolist())
heat = heat.iloc[:, global_order]
row_link = linkage(heat.values, method='average', metric='euclidean')
heat = heat.iloc[leaves_list(row_link)]

def plot_group_scatter(ax, coords, subdf, title, show_ylabel=True):
    for grp_name, color in palette.items():
        mask = subdf['serum_group'].eq(grp_name).values
        ax.scatter(coords[mask, 0], coords[mask, 1], s=24, c=color, edgecolor='black', linewidth=0.25, label=grp_name.replace(' serum', ''))
        if mask.sum() >= 3:
            pts = coords[mask, :2]
            try:
                hull = ConvexHull(pts)
                ax.add_patch(Polygon(pts[hull.vertices], closed=True, facecolor=color, alpha=0.16, edgecolor=color, lw=1.2))
            except Exception:
                pass
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(f'Dim1 ({var1:.1f}%)')
    ax.set_ylabel(f'Dim2 ({var2:.1f}%)' if show_ylabel else '')
    if not show_ylabel: ax.set_yticklabels([])

def _avoid_overlap_labels(ax, anchors, texts, *, fontsize=7.6,
                           dy_offset_px=14.0, step_px=2.4, pad_px=5.0,
                           max_iter=2000, bbox_kwargs=None, line_kwargs=None):
    bbox_kwargs = bbox_kwargs or dict(boxstyle='round,pad=0.16', fc='white',
                                       ec='none', alpha=0.85)
    line_kwargs = line_kwargs or dict(color='#666666', lw=0.4, alpha=0.65)

    fig = ax.figure
    artists = []
    for (x, y), t in zip(anchors, texts):
        disp = ax.transData.transform((x, y))
        x0, y0 = ax.transData.inverted().transform(
            (disp[0], disp[1] + dy_offset_px))
        artists.append(
            ax.text(x0, y0, t, fontsize=fontsize, ha='center', va='bottom',
                    bbox=bbox_kwargs, zorder=5)
        )
    fig.canvas.draw()

    for _ in range(max_iter):
        rdr = fig.canvas.get_renderer()
        bboxes = [a.get_window_extent(renderer=rdr).padded(pad_px) for a in artists]
        worst = None
        worst_area = 0.0
        for i in range(len(artists)):
            for j in range(i + 1, len(artists)):
                bi, bj = bboxes[i], bboxes[j]
                if bi.overlaps(bj):
                    ox = min(bi.x1, bj.x1) - max(bi.x0, bj.x0)
                    oy = min(bi.y1, bj.y1) - max(bi.y0, bj.y0)
                    area = max(0.0, ox) * max(0.0, oy)
                    if area > worst_area:
                        worst_area = area
                        worst = (i, j)
        if worst is None:
            break
        i, j = worst
        bi, bj = bboxes[i], bboxes[j]
        cix = bi.x0 + bi.width / 2
        ciy = bi.y0 + bi.height / 2
        cjx = bj.x0 + bj.width / 2
        cjy = bj.y0 + bj.height / 2
        dx, dy = cix - cjx, ciy - cjy
        mag = math.sqrt(dx * dx + dy * dy)
        if mag < 1e-6:
            dx, dy, mag = 0.0, step_px, step_px  # break degenerate stack
        # Move i and j in opposite directions, each by half the step.
        for k, sign in ((i, +1.0), (j, -1.0)):
            px, py = artists[k].get_position()
            disp = ax.transData.transform((px, py))
            nx, ny = ax.transData.inverted().transform(
                (disp[0] + sign * dx / mag * step_px,
                 disp[1] + sign * dy / mag * step_px))
            artists[k].set_position((float(nx), float(ny)))
        fig.canvas.draw()

    for (x, y), a in zip(anchors, artists):
        tx, ty = a.get_position()
        if abs(tx - x) > 1e-6 or abs(ty - y) > 1e-6:
            ax.plot([x, tx], [y, ty], zorder=1, **line_kwargs)
    return artists


def plot_volcano(ax, de_df, prefix, title, label_letter=None, show_ylabel=True):
    sig = de_df[f'{prefix}_significant']
    concord = de_df['direction_concordant']
    ax.scatter(de_df.loc[~sig, f'{prefix}_log2FC'],
               de_df.loc[~sig, f'{prefix}_neglog10_adj_p'],
               s=14, color='#d0d0d0', alpha=0.8)
    ax.scatter(de_df.loc[sig & ~concord, f'{prefix}_log2FC'],
               de_df.loc[sig & ~concord, f'{prefix}_neglog10_adj_p'],
               s=22, color='#f28e2b', alpha=0.9, label='Sig. only here')
    ax.scatter(de_df.loc[sig & concord, f'{prefix}_log2FC'],
               de_df.loc[sig & concord, f'{prefix}_neglog10_adj_p'],
               s=24, color='#c44e52', alpha=0.9, label='Direction replicated')

    top = (de_df.sort_values([f'{prefix}_adj_p_value', f'{prefix}_log2FC'],
                              ascending=[True, False])
                .head(8))
    anchors = list(zip(top[f'{prefix}_log2FC'].tolist(),
                       top[f'{prefix}_neglog10_adj_p'].tolist()))
    _avoid_overlap_labels(ax, anchors, top['protein'].tolist())

    ax.axvline(1,  color='black', ls='--', lw=0.9)
    ax.axvline(-1, color='black', ls='--', lw=0.9)
    ax.axhline(-np.log10(0.05), color='black', ls=':', lw=0.9)
    ax.set_xlabel('Log2 fold change')
    ax.set_ylabel('-log10 adjusted p-value' if show_ylabel else '')
    if not show_ylabel:
        ax.set_yticklabels([])
    ax.set_title(title, fontsize=11)
    if label_letter:
        ax.text(-0.18, 1.06, label_letter, transform=ax.transAxes,
                fontsize=16, fontweight='bold')

def plot_enrichment(ax, enr_df, cohort_name, title, selected_term, label_letter=None, show_ylabels=True):
    edf = enr_df.loc[enr_df['cohort'].eq(cohort_name)].sort_values(['adj_p_value', 'p_value', 'gene_ratio'], ascending=[True, True, False]).copy()
    y = np.arange(len(edf))
    sc = ax.scatter(edf['gene_ratio'], y, s=70 + 26*edf['overlap'], c=edf['adj_p_value'], cmap='plasma_r', edgecolor='black', linewidth=0.45)
    highlight = edf['term'].eq(selected_term)
    if highlight.any(): ax.scatter(edf.loc[highlight, 'gene_ratio'], y[highlight.values], s=240, facecolors='none', edgecolors='#111111', linewidth=1.4)
    ax.set_yticks(y)
    ax.set_yticklabels(edf['term'] if show_ylabels else ['']*len(edf), fontsize=8.8)
    ax.invert_yaxis()
    ax.set_xlabel('Gene ratio')
    ax.set_title(title, fontsize=11)
    if label_letter: ax.text(-0.18, 1.06, label_letter, transform=ax.transAxes, fontsize=16, fontweight='bold')
    return sc


def forest_panel(ax, cox_df, endpoint, selected_signature, label_letter=None):
    sub = cox_df.loc[cox_df['endpoint'].eq(endpoint)].copy()
    variables = ['age_ge_60', 'male', 'kps_ge_70', 'sig_high']
    ypos = np.arange(len(variables))[::-1]
    offsets = {'Training': 0.12, 'Test': -0.12}
    colors = {'Training': '#59a14f', 'Test': '#9c6ade'}
    ax.axvline(1, color='black', ls='--', lw=1)
    for cohort_name in ['Training', 'Test']:
        sdf = sub.loc[sub['cohort'].eq(cohort_name)].set_index('variable').reindex(variables)
        for i, var in enumerate(variables):
            r = sdf.loc[var]
            y = ypos[i] + offsets[cohort_name]
            if np.isfinite(r['HR']):
                ax.plot([r['HR_low'], r['HR_high']], [y, y], color=colors[cohort_name], lw=1.8, alpha=0.95)
                ax.scatter(r['HR'], y, s=50, color=colors[cohort_name], edgecolor='black', linewidth=0.35, zorder=3)
    ax.set_yticks(ypos)
    ax.set_yticklabels([label_map[v] for v in variables])
    ax.set_xscale('log')
    _hr_lo = pd.to_numeric(cox_df['HR_low'], errors='coerce')
    _hr_hi = pd.to_numeric(cox_df['HR_high'], errors='coerce')
    _xmin = np.nanmin(_hr_lo) / 1.6
    _xmax = np.nanmax(_hr_hi) * 1.6
    ax.set_xlim(_xmin, _xmax)
    ax.set_xticks([t for t in [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50] if _xmin <= t <= _xmax])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel('Hazard ratio')
    ax.set_title(f'{endpoint} multivariable Cox\n(selected signature: {selected_signature})', fontsize=11)
    if label_letter: ax.text(-0.18, 1.06, label_letter, transform=ax.transAxes, fontsize=16, fontweight='bold')
    handles = [Line2D([0], [0], color=colors[c], marker='o', lw=1.8, markeredgecolor='black', markeredgewidth=0.35, label=c) for c in ['Training', 'Test']]
    ax.legend(handles=handles, frameon=False, loc='lower right', fontsize=8)

### 1A — PCA of training & test cohorts (consensus PAC inset)


In [ ]:
figA, (axA1, axA2) = plt.subplots(1, 2, figsize=(9, 5))
plot_group_scatter(axA1, coords_train, train_df, 'Training', show_ylabel=True)
plot_group_scatter(axA2, coords_test, test_df, 'Test', show_ylabel=False)
axA1.legend(frameon=False, loc='lower left', fontsize=9)

# Consensus-PAC inset on the training panel
ins = axA1.inset_axes([0.12, 0.8, 0.24, 0.16])
ins.plot(metrics_df['k'], metrics_df['PAC'], marker='o', lw=1.8, color='#222222')
ins.scatter([best_k],
            [metrics_df.loc[metrics_df['k'].eq(best_k), 'PAC'].iloc[0]],
            color='#c44e52', s=28, zorder=3)
ins.set_title('Consensus PAC', fontsize=9)
ins.set_xlabel('k', fontsize=9)
ins.set_ylabel('PAC', fontsize=9)
ins.tick_params(labelsize=8)

#figA.suptitle('Figure 1A — PCA of training & test cohorts (consensus PAC inset)',
#              fontsize=12, fontweight='bold', y=1.02)
figA.tight_layout()

panel_A_path = PANEL_V2 / 'Figure1A.svg'
figA.savefig(panel_A_path, dpi=600, bbox_inches='tight')
_preview(panel_A_path)
plt.close(figA)


### 1B — annotated heatmap of top differential proteins


In [ ]:
figB = plt.figure(figsize=(14, 14.5))
gs_B = figB.add_gridspec(2, 6, height_ratios=[0.16, 1.0],
                          width_ratios=[0.09, 1.0, 0.12, 0.04, 0.03, 0.34],
                          hspace=0.06, wspace=0.04)
ann_ax, den_ax, hm_ax, cbar_ax, leg_ax = [figB.add_subplot(gs_B[r, c])
    for r, c in [(0, 1), (1, 0), (1, 1), (1, 3), (1, 5)]]

ordered_df = dc.loc[global_order].copy()
ANN_SPEC = [
    ('Cohort', ordered_df['cohort_plot'].map({'Training': 1, 'Test': 0}).values,
     {1: '#59a14f', 0: '#9c6ade'}, lambda v: 'Training' if v else 'Test'),
    ('Group', ordered_df['serum_group'].map({'Inflamed serum': 1, 'Non-inflamed serum': 0}).values,
     {1: palette['Inflamed serum'], 0: palette['Non-inflamed serum']}, lambda v: 'Inflamed' if v else 'Non-inflamed'),
    ('Selected sig. high', ordered_df['selected_signature_binary'].values,
     {1: '#c44e52', 0: '#4c78a8'}, lambda v: 'High' if v else 'Low'),
    ('Male', ordered_df['sex_completed'].astype(str).str.upper().eq('M').astype(int).values,
     {1: '#4c78a8', 0: '#e15759'}, lambda v: 'Male' if v else 'Female'),
    ('Age ≥60', (ordered_df['age_years'] >= 60).astype(int).values,
     {1: '#f28e2b', 0: '#bab0ab'}, lambda v: 'Yes' if v else 'No'),
    ('KPS ≥70', (ordered_df['Karnofsky_status'] >= 70).astype(int).values,
     {1: '#59a14f', 0: '#bab0ab'}, lambda v: 'Yes' if v else 'No'),
]
ann = pd.DataFrame({name: vals for name, vals, _, _ in ANN_SPEC}, index=ordered_df.index).T
ann_rgb = np.empty((len(ANN_SPEC), ann.shape[1], 3))
for i, (name, _, maps, _) in enumerate(ANN_SPEC):
    for j, val in enumerate(ann.loc[name].values):
        ann_rgb[i, j, :] = to_rgb(maps[int(val)])
ann_ax.imshow(ann_rgb, aspect='auto')
ann_ax.set_yticks(range(len(ann.index))); ann_ax.set_yticklabels(ann.index, fontsize=8.3)
ann_ax.set_xticks([])
for s in ann_ax.spines.values(): s.set_visible(False)

dendrogram(row_link, orientation='left', no_labels=True, color_threshold=0,
           above_threshold_color='#777777', ax=den_ax)
den_ax.invert_yaxis(); den_ax.axis('off')

sns.heatmap(heat, ax=hm_ax, cmap='RdBu_r', center=0, vmin=-2.2, vmax=2.2,
            xticklabels=False, yticklabels=True, cbar=True, cbar_ax=cbar_ax, linewidths=0.0)
hm_ax.yaxis.tick_right(); hm_ax.yaxis.set_label_position('right')
hm_ax.set_yticklabels(hm_ax.get_yticklabels(), fontsize=8.9, rotation=0)
hm_ax.tick_params(axis='y', pad=3, length=2)
cbar_ax.tick_params(labelsize=8, pad=2)
hm_ax.set_title('Top training-derived differential serum proteins (training + test)', fontsize=11, pad=9)
cbar_ax.set_ylabel('Protein z-score', fontsize=9)

train_infl = int(train_df['serum_group'].eq('Inflamed serum').sum())
train_non  = int(train_df['serum_group'].eq('Non-inflamed serum').sum())
test_infl  = int(test_df['serum_group'].eq('Inflamed serum').sum())
test_non   = int(test_df['serum_group'].eq('Non-inflamed serum').sum())
bounds = [train_infl, train_infl + train_non, train_infl + train_non + test_infl]
for x, lw, col in [(bounds[0], 1.1, '#444'), (bounds[1], 2.0, 'black'), (bounds[2], 1.1, '#444')]:
    hm_ax.axvline(x, color=col, lw=lw); ann_ax.axvline(x, color=col, lw=lw)
for xc, lab in zip([train_infl/2, train_infl + train_non/2, train_infl + train_non + test_infl/2,
                    len(global_order) - test_non/2],
                   ['Train\nInflamed', 'Train\nNon-inflamed', 'Test\nInflamed', 'Test\nNon-inflamed']):
    ann_ax.text(xc, -0.6, lab, ha='center', va='bottom', fontsize=8.6, transform=ann_ax.transData)

leg_ax.axis('off'); y = 0.98
for name, _, maps, label_fn in ANN_SPEC:
    leg_ax.text(0.22, y, name, fontsize=9.7, fontweight='bold', va='top', transform=leg_ax.transAxes)
    y -= 0.05
    for val, col in maps.items():
        leg_ax.add_patch(Rectangle((0.22, y - 0.016), 0.08, 0.022, color=col, transform=leg_ax.transAxes, clip_on=False))
        leg_ax.text(0.34, y - 0.005, label_fn(val), fontsize=8.5, va='center', transform=leg_ax.transAxes)
        y -= 0.03
    y -= 0.008

panel_B_path = PANEL_V2 / 'Figure1B.svg'
figB.savefig(panel_B_path, dpi=600, bbox_inches='tight')
_preview(panel_B_path); plt.close(figB)


### 1C — differential abundance volcanoes (Welch t, BH-FDR)


In [ ]:
figC, (axC1, axC2) = plt.subplots(1, 2, figsize=(12, 5))
plot_volcano(axC1, de, 'train', 'Training differential abundance', show_ylabel=True)
plot_volcano(axC2, de, 'test',  'Test differential abundance', show_ylabel=False)
handles, labels = axC1.get_legend_handles_labels()
if handles:
    axC2.legend(handles, labels, frameon=False, fontsize=9, loc='upper left')

figC.tight_layout()

panel_C_path = PANEL_V2 / 'Figure1C.svg'
figC.savefig(panel_C_path, dpi=600, bbox_inches='tight')
_preview(panel_C_path)
plt.close(figC)


### Figure 1D — hypergeometric enrichment of immune signatures


In [ ]:
figD, (axD1, axD2) = plt.subplots(1, 2, figsize=(12, 4.5))
sc = plot_enrichment(axD1, enr, 'Train',
                     'Training immune-signature enrichment',
                     selected_signature, show_ylabels=True)
plot_enrichment(axD2, enr, 'Test',
                'Test immune-signature enrichment',
                selected_signature, show_ylabels=False)
cb = plt.colorbar(sc, ax=[axD1, axD2], fraction=0.04, pad=0.02)
cb.set_label('Adjusted p-value')
panel_D_path = PANEL_V2 / 'Figure1D.svg'
figD.savefig(panel_D_path, dpi=600, bbox_inches='tight')
_preview(panel_D_path)
plt.close(figD)


### Figure 1E — Kaplan-Meier OS / PFS by selected signature group


In [ ]:
figE = plt.figure(figsize=(12, 10))
EGS_e = figE.add_gridspec(2, 2, hspace=0.22, wspace=0.22)
km_specs = [gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=EGS_e[r, c],
                                              height_ratios=[1.0, 0.42], hspace=0.42)
            for r in range(2) for c in range(2)]
((axE11, rE11), (axE12, rE12), (axE21, rE21), (axE22, rE22)) = [
    (figE.add_subplot(gs[0]), figE.add_subplot(gs[1])) for gs in km_specs
]

_e_group_order = [f'High {selected_signature}', f'Low {selected_signature}']
_e_times = [0, 20, 40, 60, 80]
_e_row_labels = ['High', 'Low']

def _draw_fig1e_cell(ax, rax, sub, tcol, ecol, title, *, show_ylabel, show_xlabel,
                     show_row_labels, show_header):
    plot_km(ax, sub, tcol, ecol, 'selected_signature_group',
            palette=sig_palette, group_order=_e_group_order, xmax=82,
            xlabel='Months' if show_xlabel else None,
            ylabel='Survival probability' if show_ylabel else '',
            title=title, legend_loc=None)
    ax.set_xticks(_e_times)
    if not show_ylabel:
        ax.set_yticklabels([])
    add_risk_table(rax, sub, 'selected_signature_group', tcol, times=_e_times,
                   group_order=_e_group_order, row_labels=_e_row_labels,
                   show_row_labels=show_row_labels, show_header=show_header)

_draw_fig1e_cell(axE11, rE11, train_df, 'estimated_OS_months',  'estimated_OS_event',
                 'Training OS', show_ylabel=True,  show_xlabel=False,
                 show_row_labels=True,  show_header=True)
_draw_fig1e_cell(axE12, rE12, test_df,  'estimated_OS_months',  'estimated_OS_event',
                 'Test OS',     show_ylabel=False, show_xlabel=False,
                 show_row_labels=False, show_header=False)
_draw_fig1e_cell(axE21, rE21, train_df, 'estimated_PFS_months', 'estimated_PFS_event',
                 'Training PFS', show_ylabel=True,  show_xlabel=True,
                 show_row_labels=True,  show_header=True)
_draw_fig1e_cell(axE22, rE22, test_df,  'estimated_PFS_months', 'estimated_PFS_event',
                 'Test PFS',    show_ylabel=False, show_xlabel=True,
                 show_row_labels=False, show_header=False)

legend_handles = [
    Line2D([0], [0], color=sig_palette[f'High {selected_signature}'], lw=2, label='High'),
    Line2D([0], [0], color=sig_palette[f'Low {selected_signature}'],  lw=2, label='Low'),
]
axE11.legend(handles=legend_handles, frameon=False, fontsize=9,
             loc='upper right', title=selected_signature, title_fontsize=9)

panel_E_path = PANEL_V2 / 'Figure1E.svg'
figE.savefig(panel_E_path, dpi=600, bbox_inches='tight')
_preview(panel_E_path)
plt.close(figE)

### 1F — multivariable Cox forest (selected sig)


In [ ]:
figF, (axF1, axF2) = plt.subplots(2, 1, figsize=(9, 8))
forest_panel(axF1, cox_all, 'OS',  selected_signature)
forest_panel(axF2, cox_all, 'PFS', selected_signature)

#figF.suptitle('Figure 1F — multivariable Cox forest (selected signature)',
#              fontsize=12, fontweight='bold', y=1.0)
figF.tight_layout()

panel_F_path = PANEL_V2 / 'Figure1F.svg'
figF.savefig(panel_F_path, dpi=600, bbox_inches='tight')
_preview(panel_F_path)
plt.close(figF)

### 1G — Per-protein adjusted-Cox HR volcano
Each protein is fit as a continuous predictor in a clinically-adjusted Cox proportional-hazards model

In [ ]:
cpp_dir = ROOT / 'figure1_cox_per_protein'; cpp_dir.mkdir(parents=True, exist_ok=True)
_CLIN_ADJ = ['age_ge60', 'male', 'kps_ge70']

_pp_rows = []
for ep, tcol, ecol in ENDPOINTS:
    for ch, cdf in [('train', train_df), ('test', test_df)]:
        T    = pd.to_numeric(cdf[tcol], errors='coerce').values
        E    = pd.to_numeric(cdf[ecol], errors='coerce').values
        clin = cdf[_CLIN_ADJ].apply(pd.to_numeric, errors='coerce').values
        for prot in protein_cols_all:
            x  = pd.to_numeric(cdf[prot], errors='coerce').values
            sd = np.nanstd(x)
            z  = (x - np.nanmean(x)) / sd if sd > 0 else np.full(len(x), np.nan)
            X  = np.column_stack([z, clin])
            ok = np.isfinite(T) & np.isfinite(E) & np.all(np.isfinite(X), axis=1)
            if ok.sum() < 10 or E[ok].sum() < 3:
                hr = lo = hi = p = np.nan
            else:
                f = fit_cox_hr(T[ok], E[ok], X[ok], var_names=['z_npx', *_CLIN_ADJ])
                hr, lo, hi, p = float(f['hr'][0]), float(f['lo'][0]), float(f['hi'][0]), float(f['p'][0])
            _pp_rows.append((prot, ep, ch, np.log(hr) if np.isfinite(hr) else np.nan, hr, lo, hi, p))

cox_pp = pd.DataFrame(_pp_rows, columns=['protein', 'endpoint', 'cohort', 'ln_hr', 'HR', 'HR_lo', 'HR_hi', 'p'])
cox_pp['fdr'] = np.nan
for _, sub in cox_pp.groupby(['endpoint', 'cohort']):
    m = sub['p'].notna()
    if m.any():
        cox_pp.loc[sub.index[m], 'fdr'] = multipletests(sub.loc[m, 'p'], method='fdr_bh')[1]
cox_pp.to_csv(cpp_dir / 'per_protein_cox_results.csv', index=False)
_log('Per-protein Cox fits', f'{len(protein_cols_all)} proteins x {len(ENDPOINTS)} endpoints x 2 cohorts')

In [ ]:
cox_pp = pd.read_csv(ROOT / 'figure1_cox_per_protein' / 'per_protein_cox_results.csv')

COLOR_UP, COLOR_DOWN, COLOR_NS = '#c44e52', '#4c78a8', '#bdbdbd'
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.5))
abs_lim = max(np.percentile(np.abs(cox_pp['ln_hr'].dropna()), 98), 0.5)
x_lim = (-abs_lim * 1.10, abs_lim * 1.10)

for (ep, ch), ax in zip([('OS','train'), ('OS','test'), ('PFS','train'), ('PFS','test')], axes.flat):
    other = 'test' if ch == 'train' else 'train'
    a = cox_pp[(cox_pp.endpoint==ep) & (cox_pp.cohort==ch)].set_index('protein')
    b = cox_pp[(cox_pp.endpoint==ep) & (cox_pp.cohort==other)].set_index('protein')
    a, b = a.align(b, join='inner', axis=0)
    a, b = a.dropna(subset=['ln_hr','fdr']), b.loc[a.index]

    x  = a['ln_hr'].values
    y  = -np.log10(np.clip(a['fdr'].values, 1e-300, 1.0))
    sig = a['p'].values < 0.05
    rep = (b['p'].values < 0.05) & ((a['HR'].values > 1) == (b['HR'].values > 1))
    up  = a['HR'].values > 1

    ax.scatter(np.clip(x[~sig], *x_lim), y[~sig], s=11, c=COLOR_NS, alpha=0.5, edgecolor='none')
    ax.scatter(np.clip(x[sig], *x_lim), y[sig], s=44,
               c=np.where(up[sig], COLOR_UP, COLOR_DOWN),
               edgecolor=np.where(rep[sig], 'black', 'none'),
               linewidth=np.where(rep[sig], 1.0, 0.0))
    ax.axvline(0, color='black', ls='--', lw=0.9)
    ax.axhline(-np.log10(0.05), color='grey', ls=':', lw=0.9)
    ax.set_xlim(*x_lim)
    ax.set_xlabel('ln(HR) per +1 SD NPX (adjusted)', fontsize=9)
    ax.set_ylabel('\u2212log\u2081\u2080(BH FDR)', fontsize=9)
    ax.set_title(f"{'Training' if ch=='train' else 'Test'} \u2014 {ep}  ", fontsize=10)
    ax.grid(alpha=0.2)

fig.legend(handles=[
    Line2D([0],[0], marker='o', color='none', mfc=COLOR_UP,   markersize=8, label='HR>1, p<0.05'),
    Line2D([0],[0], marker='o', color='none', mfc=COLOR_DOWN, markersize=8, label='HR<1, p<0.05'),
    Line2D([0],[0], marker='o', color='none', mfc='white', mec='black', mew=1.0, markersize=8,
           label='same-direction replicated'),
], loc='lower center', ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.03, 1, 1.0])

out = PANEL_V2 / 'Figure1G.svg'
fig.savefig(out, dpi=600, bbox_inches='tight')

### reported metrics


In [ ]:
pac_k2 = float(metrics_df.loc[metrics_df['k'].eq(2), 'PAC'].iloc[0])
_log('best_k (consensus PAC selection)', best_k)
_log('PAC at k=2', f'{pac_k2:.3f}')
_log('Training cluster sizes (Inflamed/Non-inflamed)', f'{train_infl}/{train_non}')
_log('Test cluster sizes (Inflamed/Non-inflamed)', f'{test_infl}/{test_non}')
_log('Selected signature for KM/Cox', selected_signature)

for label_, sub_, tcol_, ecol_, _ in [
    ('Innate/Inflammatory training OS',  train_df, 'estimated_OS_months',  'estimated_OS_event',  0.00262),
    ('Innate/Inflammatory training PFS', train_df, 'estimated_PFS_months', 'estimated_PFS_event', 0.00751),
    ('Innate/Inflammatory test OS',      test_df,  'estimated_OS_months',  'estimated_OS_event',  5.4e-5),
    ('Innate/Inflammatory test PFS',     test_df,  'estimated_PFS_months', 'estimated_PFS_event', 1.96e-4),
]:
    g_ = (sub_['selected_signature_group'].str.startswith('High')).astype(int).values
    _log(f'KM log-rank P — {label_}', f"{logrank_p(sub_[tcol_].values, sub_[ecol_].values, g_):.3g}")

for grp_, lbl_ in [('High', 'High Innate/Inflammatory'), ('Low', 'Low Innate/Inflammatory')]:
    sel = test_df['selected_signature_group'].str.startswith(grp_)
    s, lo, hi = km_at(test_df.loc[sel, 'estimated_OS_months'].values,
                      test_df.loc[sel, 'estimated_OS_event'].values, 24.0)
    _log(f'2-yr OS test — {lbl_}', f'{100*s:.1f}% [{100*lo:.1f}-{100*hi:.1f}]')

for grp_, lbl_ in [('High', 'High Innate/Inflammatory'), ('Low', 'Low Innate/Inflammatory')]:
    sel = test_df['selected_signature_group'].str.startswith(grp_)
    s, lo, hi = km_at(test_df.loc[sel, 'estimated_PFS_months'].values,
                      test_df.loc[sel, 'estimated_PFS_event'].values, 24.0)
    _log(f'2-yr PFS test — {lbl_}', f'{100*s:.1f}% [{100*lo:.1f}-{100*hi:.1f}]')

for endp_ in ['OS', 'PFS']:
    for coh_ in ['Training', 'Test']:
        r_ = cox_all.query("endpoint == @endp_ and cohort == @coh_ and variable == 'sig_high'").iloc[0]
        _log(f'Cox HR (sig_high) {coh_} {endp_}',
             f'{r_.HR:.2f} [{r_.HR_low:.2f}-{r_.HR_high:.2f}] P={r_.p_value:.3g}')


## 2 - Kaplan-Meier for all four immune signatures


In [ ]:
fig2_outdir = ROOT / 'figure2_km_signatures'
fig2_outdir.mkdir(exist_ok=True)
df.to_csv(fig2_outdir / 'figure2_per_signature_groups.csv', index=False)

In [ ]:
SIG_PANELS = [
    ('IFN_Chemokine', 'A'), ('Checkpoint_Cytokine', 'B'),
    ('Innate_Inflammatory', 'C'), ('Integrated_Immune', 'D'),
]
KM_PALETTE_HL = {'High': '#c44e52', 'Low': '#4c78a8'}
RISK_TIMES = [0, 12, 24, 36, 48]

def plot_fig2_panel(sig, letter):
    fig = plt.figure(figsize=(11, 11))
    gs = fig.add_gridspec(2, 2, hspace=0.25, wspace=0.22)
    gcol = f'{sig}_group'
    for endpoint, row in [('OS', 0), ('PFS', 1)]:
        tcol = 'estimated_OS_months' if endpoint == 'OS' else 'estimated_PFS_months'
        ecol = 'estimated_OS_event' if endpoint == 'OS' else 'estimated_PFS_event'
        xmax = 64 if endpoint == 'OS' else 52
        for cidx, (mask_, label) in enumerate([(is_train, 'Training'), (is_test, 'Test')]):
            km_risk_pair(fig, gs, row, cidx, df.loc[mask_], tcol, ecol, gcol,
                         times=RISK_TIMES, xmax=xmax, title=f'{label} {endpoint}',
                         palette=KM_PALETTE_HL)
    fig.suptitle(f'{sig.replace("_", "/")}  Kaplan-Meier', fontsize=14, y=0.98)
    svg = PANEL_V2 / f'Figure2{letter}.svg'
    fig.savefig(svg, dpi=600, bbox_inches='tight')
    plt.close(fig)
    return fig


In [ ]:
plot_fig2_panel('IFN_Chemokine', 'A')

In [ ]:
plot_fig2_panel('Checkpoint_Cytokine', 'B')

In [ ]:
plot_fig2_panel('Innate_Inflammatory', 'C')

In [ ]:
plot_fig2_panel('Integrated_Immune', 'D')

### log-rank P-values

In [ ]:
for sig in SIGNATURE_DEFS:
    gcol = f'{sig}_group'
    for cohort_lc, mask_ in [('training', is_train), ('validation', is_test)]:
        sub_ = df.loc[mask_]
        for ep, tcol, ecol in ENDPOINTS:
            p = logrank_p(pd.to_numeric(sub_[tcol], errors='coerce'),
                          pd.to_numeric(sub_[ecol], errors='coerce'),
                          (sub_[gcol] == 'High').astype(int).values)
            _log(f'Fig2 KM log-rank P -- {sig} {cohort_lc} {ep}', f'{p:.3g}')

## 3 - Multivariable Cox forest for all four signatures (Supp Table S2)

Each signature dichotomised at the training-cohort median and adjusted for MSKCC-like
class and male sex. Reported per signature × endpoint × cohort


In [ ]:
cf_dir = ROOT / 'multivariable_cox'
cf_dir.mkdir(exist_ok=True)

COX_SIGS = ['IFN_Chemokine', 'Checkpoint_Cytokine', 'Innate_Inflammatory',
            'Integrated_Immune', 'Immune_Apoptotic']
COX_COHORTS = [('training', '#1f77b4', is_train, -0.18, 'o'),
               ('validation', '#d62728', is_test, +0.18, 's')]

cox_rows = []

def _draw_cox_subplot(ax, sig, tcol, ecol, ename):
    ax.axvline(1, ls='--', color='black', lw=0.6)
    ylabels = [f'{sig}_high', 'MSKCC_like', 'male']
    vars_ = [f'{sig}_high', 'MSKCC_like', 'male']
    for cname, ccol, mask_, offset, marker in COX_COHORTS:
        sub = df.loc[mask_].dropna(subset=[tcol, ecol] + vars_).copy()
        X = sub[vars_].astype(float).values
        T = sub[tcol].astype(float).values
        E = sub[ecol].astype(int).values
        fit = fit_cox_hr(T, E, X, var_names=vars_)
        for k in range(len(vars_)):
            y = k + offset
            ax.plot([fit['lo'][k], fit['hi'][k]], [y, y], color=ccol, lw=1.2)
            ax.plot(fit['hr'][k], y, marker, color=ccol, ms=6, markeredgecolor='black')
            star = '*' if (np.isfinite(fit['p'][k]) and fit['p'][k] < 0.05) else ''
            ax.text(max(fit['hi'][k], 1.05), y, f'{fit["hr"][k]:.2f}{star}',
                    fontsize=7, va='center', color=ccol)
            cox_rows.append({'signature': sig, 'cohort': cname, 'endpoint': ename,
                             'covariate': ylabels[k],
                             'HR': fit['hr'][k], 'lo': fit['lo'][k],
                             'hi': fit['hi'][k], 'p': fit['p'][k]})
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.set_xscale('log')
    ax.set_xlim(0.2, 10)
    ax.invert_yaxis()
    ax.set_title(f'{sig.replace("_", "/")} — {ename}', fontsize=10)
    ax.set_xlabel('Hazard ratio (log scale)')

fig, axes = plt.subplots(len(COX_SIGS), 2, figsize=(11, 3.5 * len(COX_SIGS)),
                         sharey='row', squeeze=False)

for row, sig in enumerate(COX_SIGS):
    med_train = df.loc[is_train, sig].median()
    df[f'{sig}_high'] = (df[sig] >= med_train).astype(int)
    _draw_cox_subplot(axes[row, 0], sig, 'estimated_OS_months',  'estimated_OS_event',  'OS')
    _draw_cox_subplot(axes[row, 1], sig, 'estimated_PFS_months', 'estimated_PFS_event', 'PFS')

handles = [Line2D([0], [0], color='#1f77b4', marker='o', lw=1.2, label='training'),
           Line2D([0], [0], color='#d62728', marker='s', lw=1.2, label='test')]
fig.legend(handles=handles, loc='upper center', ncol=2, frameon=False,
           bbox_to_anchor=(0.5, 1.01))
fig.suptitle('Multivariable Cox — immune signatures', fontsize=12, y=1.02)
fig.tight_layout()

svg = PANEL_V2 / 'Figure3.svg'
fig.savefig(svg, dpi=600, bbox_inches='tight')

pd.DataFrame(cox_rows).to_csv(cf_dir / 'cox_results.csv', index=False)
_log('Cox forest saved', 'single stacked Figure3.svg/.pdf')

In [ ]:
_table_s2 = pd.DataFrame(cox_rows)
for sig in ['IFN_Chemokine', 'Checkpoint_Cytokine', 'Innate_Inflammatory', 'Integrated_Immune']:
    for ep in ['OS', 'PFS']:
        for coh in ['training', 'validation']:
            cov_name = f'{sig}_high'
            r = _table_s2[(_table_s2['signature'] == sig) & (_table_s2['endpoint'] == ep) & (_table_s2['cohort'] == coh) & (_table_s2['covariate'] == cov_name)]
            if r.empty: continue
            r = r.iloc[0]
            _log(f'Cox HR — {sig} {coh} {ep}',
                       f'{r.HR:.2f} [{r.lo:.2f}-{r.hi:.2f}] P={r.p:.3g}')

In [ ]:
# True univariate Cox HRs (sig_high alone, no covariates)
_univ_rows = []
for _sig in ['IFN_Chemokine', 'Checkpoint_Cytokine', 'Innate_Inflammatory',
             'Integrated_Immune', 'Immune_Apoptotic']:
    _med = df.loc[is_train, _sig].median()
    df[f'{_sig}_high_train_split'] = (df[_sig] >= _med).astype(int)
    for _cohort, _mask in [('training', is_train), ('validation', is_test)]:
        for _ep, _tcol, _ecol in ENDPOINTS:
            _ss = df.loc[_mask].dropna(subset=[_tcol, _ecol,
                                                f'{_sig}_high_train_split']).copy()
            _T = pd.to_numeric(_ss[_tcol],  errors='coerce').values.astype(float)
            _E = pd.to_numeric(_ss[_ecol],  errors='coerce').values.astype(int)
            _X = _ss[[f'{_sig}_high_train_split']].astype(float).values
            f_ = fit_cox_hr(_T, _E, _X)
            _hr, _lo, _hi, _p = float(f_['hr'][0]), float(f_['lo'][0]), float(f_['hi'][0]), float(f_['p'][0])
            _univ_rows.append({'signature': _sig, 'cohort': _cohort, 'endpoint': _ep,
                               'HR': _hr, 'lo': _lo, 'hi': _hi, 'p': _p})
            _log(f'Univariate Cox HR — {_sig} {_cohort} {_ep}',
                 f'{_hr:.2f} [{_lo:.2f}-{_hi:.2f}] P={_p:.3g}')

## Sensitivity Cox Innate/Inflammatory(Supp Table S3)

In [ ]:
sens_dir = ROOT / 'sensitivity'
sens_dir.mkdir(exist_ok=True)

prot_z_cols = [c for c in df.columns if c.startswith('z_')]
zmat = df[prot_z_cols].fillna(0.0).values
sample_iqr = np.percentile(zmat, 75, axis=1) - np.percentile(zmat, 25, axis=1)
sample_sd  = zmat.std(axis=1, ddof=0)
_corr = np.corrcoef(zmat); np.fill_diagonal(_corr, np.nan)
mean_corr = np.nanmean(_corr, axis=1)
def _robust_z(x):
    med = np.nanmedian(x); mad = np.nanmedian(np.abs(x - med))
    s = mad if mad > 0 else np.nanstd(x)
    return 0.6745 * (x - med) / s if s > 0 else np.zeros_like(x)
df['outlier_score'] = (np.maximum(_robust_z(sample_iqr), 0)
                       + np.maximum(_robust_z(sample_sd), 0)
                       + np.maximum(_robust_z(-mean_corr), 0))

sig = 'Innate_Inflammatory'
med_train = df.loc[is_train, sig].median()
df[f'{sig}_high'] = (df[sig] >= med_train).astype(int)
COVARS = [f'{sig}_high', 'MSKCC_like', 'male']

def _fit_sens(sub, tcol, ecol):
    sub = sub.dropna(subset=[tcol, ecol] + COVARS).copy()
    if len(sub) < 20: return None
    X = sub[COVARS].astype(float).values
    T = sub[tcol].astype(float).values
    E = sub[ecol].astype(int).values
    f_ = fit_cox_hr(T, E, X)
    if f_['model'] is None: return None
    return {'n': int(len(sub)), 'events': int(E.sum()),
            'HR': float(f_['hr'][0]), 'CI_lo': float(f_['lo'][0]),
            'CI_hi': float(f_['hi'][0]), 'P': float(f_['p'][0])}

_age = pd.to_numeric(df['age_years'],        errors='coerce')
_kps = pd.to_numeric(df['Karnofsky_status'], errors='coerce')
restrictions = {
    'full':              lambda d: d,
    'KPS>=70':           lambda d: d[_kps.loc[d.index] >= 70],
    'excl_top4_outlier': lambda d: d.nsmallest(len(d) - 4, 'outlier_score'),
    'age>=60':           lambda d: d[_age.loc[d.index] >= 60],
    'age<60':            lambda d: d[_age.loc[d.index] < 60],
}

rows = []
for cohort_lbl, cohort_mask in [('training', is_train), ('validation', is_test)]:
    cohort_df = df.loc[cohort_mask]
    for rname, restrict in restrictions.items():
        sub = restrict(cohort_df)
        for ename, tcol, ecol in ENDPOINTS:
            r = _fit_sens(sub, tcol, ecol)
            if r is None: continue
            rows.append({'cohort': cohort_lbl, 'restriction': rname,
                         'endpoint': ename, **r})

sens_df = pd.DataFrame(rows)
sens_df.to_csv(sens_dir / 'sensitivity_cox.csv', index=False)
print(f'Sensitivity Cox fits: {len(sens_df)} models written -> {sens_dir / "sensitivity_cox.csv"}')

In [ ]:
for _, r in sens_df.iterrows():
    _log(f'Sens Cox — {r.restriction} {r.cohort} {r.endpoint}',
               f'HR={r.HR:.2f} [{r.CI_lo:.2f}-{r.CI_hi:.2f}] P={r.P:.3g} n={r.n}/{r.events}')

## 4 - Harrell C-index benchmarking (Supp Table S1)

9 Cox models (3 single-clinical, MSKCC-like, clinical baseline, 4 clinical + signature) ×
OS / PFS × training / test.


In [ ]:
fig4_outdir = ROOT / 'figure4_cindex'
fig4_outdir.mkdir(exist_ok=True)

train = df.loc[is_train].copy().reset_index(drop=True)
test  = df.loc[is_test ].copy().reset_index(drop=True)

In [ ]:
def harrell_c_index(time, event, risk):
    t = np.asarray(time, float)
    e = np.asarray(event, int)
    r = np.asarray(risk, float)
    conc = disc = tied = comp = 0
    n = len(t)
    for i in range(n):
        for j in range(i + 1, n):
            if np.isnan(t[i]) or np.isnan(t[j]) or np.isnan(r[i]) or np.isnan(r[j]):
                continue
            if t[i] == t[j]:
                continue
            if t[i] < t[j] and e[i] == 1:
                comp += 1
                if r[i] > r[j]:
                    conc += 1
                elif r[i] < r[j]:
                    disc += 1
                else:
                    tied += 1
            elif t[j] < t[i] and e[j] == 1:
                comp += 1
                if r[j] > r[i]:
                    conc += 1
                elif r[j] < r[i]:
                    disc += 1
                else:
                    tied += 1
    return (conc + 0.5 * tied) / comp if comp > 0 else np.nan

In [ ]:
def fit_and_score(train_df, test_df, duration_col, event_col, features):
    cols = list(features) + [duration_col, event_col]
    tr = train_df[cols].apply(pd.to_numeric, errors='coerce').dropna().copy()
    te = test_df[cols].apply(pd.to_numeric, errors='coerce').dropna().copy()
    ex_tr = tr[features]
    ex_te = te[features]
    model = PHReg(tr[duration_col], ex_tr, status=tr[event_col])
    fit = model.fit(disp=0)
    beta = np.asarray(fit.params, float)
    risk_tr = np.dot(ex_tr.values, beta)
    risk_te = np.dot(ex_te.values, beta)
    return {
        'n_train': len(tr), 'n_test': len(te),
        'train_cindex': harrell_c_index(tr[duration_col], tr[event_col], risk_tr),
        'test_cindex': harrell_c_index(te[duration_col], te[event_col], risk_te),
        'beta': beta.tolist(),
    }

model_defs = [
    ('Age ≥ 60', ['age_ge60'], 'Single clinical', 'Age'),
    ('KPS ≥ 70', ['kps_ge70'], 'Single clinical', 'KPS'),
    ('Male sex', ['male'], 'Single clinical', 'Sex'),
    ('MSKCC-like', ['MSKCC_like'], 'Clinical score', 'MSKCC'),
    ('Clinical', ['age_ge60', 'kps_ge70', 'male'], 'Clinical baseline', 'Clinical'),
    ('Clinical + IFN/Chemokine', ['age_ge60', 'kps_ge70', 'male', 'IFN_Chemokine'], 'Clinical + signature', 'Clin+IFN'),
    ('Clinical + Checkpoint/Cytokine', ['age_ge60', 'kps_ge70', 'male', 'Checkpoint_Cytokine'], 'Clinical + signature', 'Clin+Chk'),
    ('Clinical + Innate/Inflammatory', ['age_ge60', 'kps_ge70', 'male', 'Innate_Inflammatory'], 'Clinical + signature', 'Clin+Innate'),
    ('Clinical + Integrated Immune', ['age_ge60', 'kps_ge70', 'male', 'Integrated_Immune'], 'Clinical + signature', 'Clin+Int'),
    ('IFN/Chemokine only', ['IFN_Chemokine'], 'Signature only', 'Sig-IFN'),
    ('Checkpoint/Cytokine only', ['Checkpoint_Cytokine'], 'Signature only', 'Sig-Chk'),
    ('Innate/Inflammatory only', ['Innate_Inflammatory'], 'Signature only', 'Sig-Innate'),
    ('Integrated Immune only', ['Integrated_Immune'], 'Signature only', 'Sig-Int')]

rows = []
for endpoint_name, duration_col, event_col in [('OS', 'estimated_OS_months', 'estimated_OS_event'), ('PFS', 'estimated_PFS_months', 'estimated_PFS_event')]:
    for model_name, features, category, short_label in model_defs:
        res = fit_and_score(train, test, duration_col, event_col, features)
        rows.append({'endpoint': endpoint_name, 'model': model_name, 'short_label': short_label, 'category': category, 'features': ';'.join(features), **res})

metrics = pd.DataFrame(rows)
for endpoint in ['OS', 'PFS']:
    base = metrics[(metrics['endpoint'] == endpoint) & (metrics['model'] == 'Clinical')].iloc[0]
    mask = metrics['endpoint'] == endpoint
    metrics.loc[mask, 'delta_train_vs_clinical'] = metrics.loc[mask, 'train_cindex'] - base['train_cindex']
    metrics.loc[mask, 'delta_test_vs_clinical'] = metrics.loc[mask, 'test_cindex'] - base['test_cindex']

metrics.to_csv(fig4_outdir / 'cindex_model_comparison_metrics_notebook.csv', index=False)

category_colors = {'Single clinical': '#8f8f8f', 'Clinical score': '#4c78a8', 'Clinical baseline': '#000000', 'Clinical + signature': '#e69f00', 'Signature only': '#2ca02c', 'Proteome-wide': '#9467bd'}
signature_offsets = {'Clin+IFN': (0.004, 0.004), 'Clin+Chk': (0.004, -0.006), 'Clin+Innate': (0.004, 0.008), 'Clin+Int': (0.004, -0.010), 'Clinical': (0.004, 0.004), 'MSKCC': (0.004, 0.004), 'Age': (0.004, -0.006), 'KPS': (0.004, 0.006), 'Sex': (0.004, -0.010), 'Sig-IFN': (0.004, 0.004), 'Sig-Chk': (0.004, -0.006), 'Sig-Innate': (0.004, 0.008), 'Sig-Int': (0.004, -0.010), 'EN-prot': (0.004, 0.004)}

In [ ]:
ENET_CACHE     = fig4_outdir / 'enet_cox_proteome_cache.csv'
ENET_SELECTED  = fig4_outdir / 'enet_cox_selected_proteins.csv'
RECOMPUTE_ENET = False
ENET_GRID_ALPHA = [0.1, 0.3, 0.6, 1.0]
ENET_GRID_L1    = [0.5, 0.9]           
ENET_KFOLDS     = 3

def _enet_prep(frame, cols, med, mu, sd):
    # median-impute missing NPX, then z-score so the penalty treats every protein equally
    X = frame[cols].apply(pd.to_numeric, errors='coerce').fillna(med)
    return ((X - mu) / sd).values

def _enet_fit_score(tr_df, te_df, dur, evt, cols, alpha, l1):
    tr = tr_df[[dur, evt]].apply(pd.to_numeric, errors='coerce'); keep = tr.dropna().index
    Xraw = tr_df.loc[keep, cols].apply(pd.to_numeric, errors='coerce')
    med = Xraw.median(); Xf = Xraw.fillna(med); mu = Xf.mean(); sd = Xf.std(ddof=0).replace(0, 1.0)
    Xtr = _enet_prep(tr_df.loc[keep], cols, med, mu, sd)
    ttr = tr.loc[keep, dur].values; etr = tr.loc[keep, evt].values.astype(int)
    res = PHReg(ttr, Xtr, status=etr).fit_regularized(method='elastic_net', alpha=alpha, L1_wt=l1)
    beta = np.asarray(res.params, float)
    te = te_df[[dur, evt]].apply(pd.to_numeric, errors='coerce'); keept = te.dropna().index
    Xte = _enet_prep(te_df.loc[keept], cols, med, mu, sd)
    return dict(train_cindex=harrell_c_index(ttr, etr, Xtr @ beta),
                test_cindex=harrell_c_index(te.loc[keept, dur].values, te.loc[keept, evt].values, Xte @ beta),
                n_nonzero=int((beta != 0).sum()), n_train=len(keep), n_test=len(keept), beta=beta)

def _enet_tune(tr_df, dur, evt, cols):
    # choose (alpha, l1) by mean held-out C-index over inner kfols
    tr = tr_df[[dur, evt]].apply(pd.to_numeric, errors='coerce'); idx = np.array(tr.dropna().index)
    kf = KFold(n_splits=ENET_KFOLDS, shuffle=True, random_state=42); best = (-1.0, None)
    for a in ENET_GRID_ALPHA:
        for l in ENET_GRID_L1:
            cs = []
            for tri, vai in kf.split(idx):
                try:
                    r = _enet_fit_score(tr_df.loc[idx[tri]], tr_df.loc[idx[vai]], dur, evt, cols, a, l)
                    if not np.isnan(r['test_cindex']): cs.append(r['test_cindex'])
                except Exception:
                    pass
            if cs and np.mean(cs) > best[0]:
                best = (float(np.mean(cs)), (a, l))
    return best

if RECOMPUTE_ENET or not ENET_CACHE.exists():
    _enet_rows, _sel_rows = [], []
    for ep, dur, evt in [('OS', 'estimated_OS_months', 'estimated_OS_event'),
                         ('PFS', 'estimated_PFS_months', 'estimated_PFS_event')]:
        best_cv, (a, l) = _enet_tune(train, dur, evt, protein_cols_all)
        r = _enet_fit_score(train, test, dur, evt, protein_cols_all, a, l)
        _enet_rows.append(dict(endpoint=ep, model='Proteome-wide elastic-net Cox', short_label='EN-prot',
            category='Proteome-wide', features=f'proteome ({len(protein_cols_all)})',
            n_train=r['n_train'], n_test=r['n_test'], train_cindex=r['train_cindex'],
            test_cindex=r['test_cindex'], best_cv=best_cv, alpha=a, l1_ratio=l, n_nonzero=r['n_nonzero']))
        for prot, b in zip(protein_cols_all, r['beta']):
            if b != 0:
                _sel_rows.append(dict(endpoint=ep, protein=prot, coef=float(b)))
        _log(f'enet-Cox {ep}', f"test C={r['test_cindex']:.3f} ({r['n_nonzero']} proteins, alpha={a}, l1={l})")
    enet_cache = pd.DataFrame(_enet_rows); enet_cache.to_csv(ENET_CACHE, index=False)
    pd.DataFrame(_sel_rows).to_csv(ENET_SELECTED, index=False)
else:
    enet_cache = pd.read_csv(ENET_CACHE)
    for _, r in enet_cache.iterrows():
        _log(f'enet-Cox {r.endpoint} (cached)', f"test C={r.test_cindex:.3f} ({int(r.n_nonzero)} proteins)")

metrics = metrics[metrics['category'] != 'Proteome-wide'].copy()
metrics = pd.concat([metrics, enet_cache[['endpoint', 'model', 'short_label', 'category', 'features',
            'n_train', 'n_test', 'train_cindex', 'test_cindex']]], ignore_index=True)
for endpoint in ['OS', 'PFS']:
    base = metrics[(metrics['endpoint'] == endpoint) & (metrics['model'] == 'Clinical')].iloc[0]
    m = metrics['endpoint'] == endpoint
    metrics.loc[m, 'delta_train_vs_clinical'] = metrics.loc[m, 'train_cindex'] - base['train_cindex']
    metrics.loc[m, 'delta_test_vs_clinical']  = metrics.loc[m, 'test_cindex']  - base['test_cindex']
metrics.to_csv(fig4_outdir / 'cindex_model_comparison_metrics_notebook.csv', index=False)


In [ ]:
def _save_fig(fig, letter, slug):
    fig.tight_layout()
    save_panel(fig, f'Figure4{letter}_{slug}', copy=False, preview=False, close=False)
    return fig

def _plot_cindex_scatter(endpoint, letter):
    fig, ax = plt.subplots(figsize=(8, 6.6))   # was (6.2, 5.2)
    sub = metrics[metrics['endpoint'] == endpoint].copy()
    for cat in ['Single clinical', 'Clinical score', 'Clinical baseline', 'Signature only', 'Clinical + signature', 'Proteome-wide']:
        part = sub[sub['category'] == cat]
        ax.scatter(part['train_cindex'], part['test_cindex'],
                   s=85 if cat != 'Clinical baseline' else 110,
                   color=category_colors[cat],
                   edgecolor='black' if cat == 'Clinical baseline' else 'none',
                   alpha=0.95, label=cat, zorder=3)
    _v = pd.concat([sub['train_cindex'], sub['test_cindex']]).dropna()
    lims = [min(0.45, float(_v.min()) - 0.02), max(0.76, float(_v.max()) + 0.03)]
    ax.plot(lims, lims, linestyle='--', linewidth=1.2, color='0.5', zorder=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Training C-index'); ax.set_ylabel('Test C-index')
    ax.set_title(f'{endpoint}: training vs test C-index')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
    anchors = list(zip(sub['train_cindex'].tolist(), sub['test_cindex'].tolist()))
    _avoid_overlap_labels(
        ax, anchors, sub['short_label'].tolist(),
        fontsize=9, dy_offset_px=16.0, step_px=2.8, pad_px=6.0,
        bbox_kwargs=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.9),
    )
    return fig

def _plot_cindex_delta(endpoint, letter):
    fig, ax = plt.subplots(figsize=(7.4, 5.0))
    sub = (metrics[metrics['endpoint'] == endpoint]
           .copy()
           .sort_values(['delta_test_vs_clinical', 'delta_train_vs_clinical'],
                        ascending=[True, True]).reset_index(drop=True))
    y = np.arange(len(sub))
    for i, (_, rr) in enumerate(sub.iterrows()):
        ax.plot([rr['delta_train_vs_clinical'], rr['delta_test_vs_clinical']],
                [i, i], color='0.75', linewidth=1.6, zorder=1)
        ax.scatter(rr['delta_train_vs_clinical'], i, s=75,
                   color=category_colors[rr['category']], zorder=3)
        ax.scatter(rr['delta_test_vs_clinical'], i, s=75, facecolor='white',
                   edgecolor=category_colors[rr['category']], linewidth=1.8, zorder=4)
    ax.axvline(0.0, linestyle='--', linewidth=1.2, color='0.4')
    ax.set_yticks(y); ax.set_yticklabels(sub['model'])
    ax.set_xlabel('Δ C-index vs clinical baseline')
    ax.set_title(f'{endpoint}: improvement over clinical baseline')
    ax.grid(axis='x', alpha=0.25)
    legend_handles = [
        plt.Line2D([0], [0], marker='o', color='none', markerfacecolor='0.4',
                   markeredgecolor='0.4', markersize=8, label='Training'),
        plt.Line2D([0], [0], marker='o', color='none', markerfacecolor='white',
                   markeredgecolor='0.4', markersize=8, label='Test'),
    ]
    ax.legend(handles=legend_handles, frameon=False, fontsize=8, loc='lower right')
    return fig

_figs6 = [
    _save_fig(_plot_cindex_scatter('OS', 'A'),  'A', 'OS_train_test_scatter'),
    _save_fig(_plot_cindex_scatter('PFS', 'B'), 'B', 'PFS_train_test_scatter'),
    _save_fig(_plot_cindex_delta('OS', 'C'),    'C', 'OS_delta_vs_clinical'),
    _save_fig(_plot_cindex_delta('PFS', 'D'),   'D', 'PFS_delta_vs_clinical')]

In [ ]:
for _, r in metrics.iterrows():
    _log(f'C-index — {r.endpoint} {r.short_label}',
               f'train={r.train_cindex:.2f} test={r.test_cindex:.2f}')

for ep in ['OS', 'PFS']:
    deltas = metrics[(metrics['endpoint'] == ep) & (metrics['category'] == 'Clinical + signature')]['delta_test_vs_clinical']
    _log(f'ΔC-index test {ep} (clin+sig vs clin)',
               f'min={deltas.min():+.2f} max={deltas.max():+.2f}')

## 5 - Binary-endpoint discrimination across feature sets

Predicting two baseline binary outcomes — **bad responder** (PFS event < 9 months) and **long survivor** (OS > 18 months)


In [ ]:
binbench_dir = ROOT / 'binary_benchmark'; binbench_dir.mkdir(exist_ok=True)

_CLIN = ['male', 'age_ge60', 'kps_ge70']
_SIG  = list(SIGNATURE_DEFS)
BIN_FEATURE_SETS = {
    'Clinical':            _CLIN,
    'Signature-only':      _SIG,
    'Clinical+signature':  _CLIN + _SIG,
    'Proteome-wide':       protein_cols_all}
BIN_ENDPOINTS = {
    'Bad responder (PFS<9mo)': 'bad_responder',
    'Long survival (OS>18mo)': 'long_survivor'}

def _bin_pipe(proteome):

    steps = [('imp', SimpleImputer(strategy='median'))]
    if proteome:
        steps.append(('sel', SelectKBest(f_classif, k=100)))
    steps += [('sc', StandardScaler()),
              ('clf', LogisticRegression(penalty='elasticnet', solver='saga',
                                         class_weight='balanced', max_iter=5000, random_state=42))]
    grid = {'clf__C': [0.05, 0.2, 1.0], 'clf__l1_ratio': [0.5, 0.9]}
    if proteome:
        grid['sel__k'] = [50, 100]
    return Pipeline(steps), grid

_bin_rows = []
bin_pred  = {}
for elabel, ecol in BIN_ENDPOINTS.items():
    yfull = pd.to_numeric(df[ecol], errors='coerce') # may contain NaN
    valid = yfull.notna()
    tr_mask = (is_train & valid).values
    te_mask = (is_test  & valid).values
    pooled  = df.loc[valid.values]
    y_pool  = yfull[valid].astype(int)
    for fname, feats in BIN_FEATURE_SETS.items():
        proteome = fname == 'Proteome-wide'
        pipe, grid = _bin_pipe(proteome)
     
        gs = GridSearchCV(pipe, grid, scoring='roc_auc',
                          cv=StratifiedKFold(5, shuffle=True, random_state=1), n_jobs=-1)
        gs.fit(df.loc[tr_mask, feats], yfull[tr_mask].astype(int))
        y_te = yfull[te_mask].astype(int)
        p_te = gs.predict_proba(df.loc[te_mask, feats])[:, 1]
        held = roc_auc_score(y_te, p_te)
        bin_pred[(elabel, fname)] = (y_te.to_numpy(), p_te)

        gs2 = GridSearchCV(pipe, grid, scoring='roc_auc',
                           cv=StratifiedKFold(4, shuffle=True, random_state=3), n_jobs=-1)
        nested = cross_val_score(gs2, pooled[feats], y_pool, scoring='roc_auc',
                                 cv=StratifiedKFold(5, shuffle=True, random_state=7), n_jobs=1)
        _bin_rows.append(dict(endpoint=elabel, feature_set=fname, n_valid=int(valid.sum()),
                              held_out_auc=held, nestedcv_auc=nested.mean(), nestedcv_sd=nested.std()))
        _log(f'binary AUC {elabel} {fname}', f'held-out={held:.3f} nestedCV={nested.mean():.3f}')

    rf = Pipeline([('imp', SimpleImputer(strategy='median')),
                   ('sel', SelectKBest(f_classif, k=40)),
                   ('clf', RandomForestClassifier(n_estimators=800, max_features='sqrt',
                        class_weight='balanced_subsample', min_samples_split=4, min_samples_leaf=2,
                        random_state=42, n_jobs=-1))])
    rf.fit(df.loc[tr_mask, protein_cols_all], yfull[tr_mask].astype(int))
    rf_auc = roc_auc_score(yfull[te_mask].astype(int),
                           rf.predict_proba(df.loc[te_mask, protein_cols_all])[:, 1])
    _bin_rows.append(dict(endpoint=elabel, feature_set='Proteome-wide RF (sensitivity)',
                          n_valid=int(valid.sum()), held_out_auc=rf_auc,
                          nestedcv_auc=np.nan, nestedcv_sd=np.nan))
    _log(f'binary AUC {elabel} RF sensitivity', f'held-out={rf_auc:.3f} (n_valid={int(valid.sum())})')

binary_bench = pd.DataFrame(_bin_rows)
binary_bench.to_csv(binbench_dir / 'binary_benchmark_auc.csv', index=False)
binary_bench

In [ ]:
_eps       = list(BIN_ENDPOINTS)
_bar_order = ['Clinical', 'Signature-only', 'Clinical+signature', 'Proteome-wide']
_roc_sets  = ['Clinical', 'Signature-only', 'Clinical+signature']
_bin_colors = {'Clinical': '#000000', 'Signature-only': '#2ca02c',
               'Clinical+signature': '#e69f00', 'Proteome-wide': '#9467bd'}

fig5, axes = plt.subplots(2, 2, figsize=(13, 9.5))
for j, ep in enumerate(_eps):

    sub = binary_bench[binary_bench['endpoint'] == ep].set_index('feature_set').reindex(_bar_order)
    x = np.arange(len(_bar_order)); w = 0.38; ax = axes[0, j]
    ax.bar(x - w/2, sub['held_out_auc'], w, color=[_bin_colors[f] for f in _bar_order],
           edgecolor='black', linewidth=0.6, label='Held-out (train->test)')
    ax.bar(x + w/2, sub['nestedcv_auc'], w, yerr=sub['nestedcv_sd'], capsize=3,
           color=[_bin_colors[f] for f in _bar_order], edgecolor='black', linewidth=0.6,
           alpha=0.55, hatch='//', label='Nested CV (pooled)')
    ax.axhline(0.5, ls='--', lw=1, color='0.5'); ax.set_xticks(x)
    ax.set_xticklabels(['Clinical', 'Signature', 'Clin+Sig', 'Proteome'], rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0.4, 0.85); ax.set_title(f'{ep}\nAUC by feature set'); ax.grid(axis='y', alpha=0.25)
    if j == 0:
        ax.set_ylabel('ROC AUC'); ax.legend(frameon=False, fontsize=8, loc='upper right')

    axr = axes[1, j]
    for fs in _roc_sets:
        y_te, p_te = bin_pred[(ep, fs)]
        fpr, tpr, _ = roc_curve(y_te, p_te); a = roc_auc_score(y_te, p_te)
        axr.plot(fpr, tpr, color=_bin_colors[fs], lw=2, label=f'{fs} (AUC {a:.2f})')
    axr.plot([0, 1], [0, 1], ls='--', color='0.6', lw=1)
    axr.set_xlim(0, 1); axr.set_ylim(0, 1.02); axr.set_xlabel('False positive rate')
    axr.set_title(f'{ep}\nROC \u2014 held-out test'); axr.legend(frameon=False, fontsize=8, loc='lower right')
    if j == 0: axr.set_ylabel('True positive rate')

fig5.tight_layout(rect=[0, 0, 1, 0.975])

fig5.savefig(PANEL_V2 / 'Figure5.svg', bbox_inches='tight')
plt.show()


## Supplementary S1 — cohort and batch-effect

Shows the batch shift between training and test runs


In [ ]:
be_dir = ROOT / 'supp_batch_effect'
be_dir.mkdir(exist_ok=True)

raw = pd.read_csv(REPLICATES_CSV)
raw = raw[~raw['patient_id'].astype(str).str.contains('/')].copy()
meta_be = ['patient_id','MRI','omics_ID','sex_completed','age_years','cohort',
           'Karnofsky_status','Karnofsky_group','estimated_OS_months','estimated_OS_event',
           'estimated_PFS_months','estimated_PFS_event']
prot_cols_be = [c for c in raw.columns if c not in meta_be]
X = raw[prot_cols_be].apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.median())
cohort = raw['cohort'].astype(str).str.lower()

Xs = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
Xs = Xs.fillna(0.0)
pca = PCA(n_components=2).fit_transform(Xs.values)

t_rows = []
tr = (cohort == 'training').values
te = (cohort == 'validation').values
for c in prot_cols_be:
    a = X.loc[tr, c].dropna().values
    b = X.loc[te, c].dropna().values
    if len(a) > 5 and len(b) > 5:
        t, p = stats.ttest_ind(a, b, equal_var=False)
        t_rows.append({'protein': c, 't': t, 'p': p,
                       'mean_train': float(np.mean(a)), 'mean_test': float(np.mean(b))})
batch_df = pd.DataFrame(t_rows)
batch_df['fdr'] = multipletests(batch_df['p'], method='fdr_bh')[1]
batch_df.to_csv(be_dir / 'per_protein_cohort_diff.csv', index=False)

panel_size = (5.2, 4.6)

fig_a, ax = plt.subplots(figsize=panel_size)
for grp, col in [('training', '#1f77b4'), ('validation', '#d62728')]:
    sel = (cohort == grp).values
    ax.scatter(pca[sel, 0], pca[sel, 1], s=22, alpha=0.7, color=col,
               label=grp, edgecolor='black', lw=0.3)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Raw-NPX PCA by cohort')
ax.legend(frameon=False, fontsize=8)
fig_a.tight_layout()
path_a = PANEL_V2 / 'supp_fig_s1_a_pca.svg'
fig_a.savefig(path_a, dpi=600, bbox_inches='tight')

fig_b, ax = plt.subplots(figsize=panel_size)
medians = X.median(axis=1)
for grp, col in [('training', '#1f77b4'), ('validation', '#d62728')]:
    sel = (cohort == grp).values
    ax.hist(medians[sel], bins=18, alpha=0.55, label=grp, color=col,
            edgecolor='black', lw=0.3)
ax.set_xlabel('Per-sample median NPX')
ax.set_ylabel('Samples')
ax.set_title('Median NPX distribution by cohort')
ax.legend(frameon=False, fontsize=8)
fig_b.tight_layout()
path_b = PANEL_V2 / 'supp_fig_s1_b_median.svg'
fig_b.savefig(path_b, dpi=600, bbox_inches='tight')

fig_c, ax = plt.subplots(figsize=panel_size)
ax.scatter(np.log2(np.abs(batch_df['mean_train'] - batch_df['mean_test']) + 1e-3),
           -np.log10(batch_df['p']), s=8, alpha=0.5, color='#444')
ax.axhline(-np.log10(0.05), ls='--', color='red', lw=0.6)
ax.set_xlabel('|mean(train) - mean(test)|  (log2)')
ax.set_ylabel('-log10 raw P')
ax.set_title(f"Per-protein train vs test\n"
             f"{(batch_df['fdr']<0.05).sum()}/{len(batch_df)} differ at FDR<0.05")
fig_c.tight_layout()
path_c = PANEL_V2 / 'supp_fig_s1_c_volcano.svg'
fig_c.savefig(path_c, dpi=600, bbox_inches='tight')


In [ ]:
n_diff = int((batch_df['fdr'] < 0.05).sum())
_log('Proteins differing train vs test (FDR<0.05)', f'{n_diff}/{len(batch_df)}')

## Supplementary S2 — heatmap variable prots

Top-150 proteins selected from TRAINING NPX variance


In [ ]:
ch_dir = ROOT / 'supp_corr_heatmap'
ch_dir.mkdir(exist_ok=True)

df_tr_s2 = df[df['cohort'] == 'training']
raw_tr_s2 = df_tr_s2[protein_cols_all].apply(pd.to_numeric, errors='coerce')
variances_tr = raw_tr_s2.var().sort_values(ascending=False)
top_proteins = variances_tr.head(150).index.tolist()

corr_tr_for_order = raw_tr_s2[top_proteins].corr(method='spearman')
dist = 1 - corr_tr_for_order.abs()
order_idx = leaves_list(linkage(dist.values[np.triu_indices_from(dist.values, k=1)], method='average'))
top_order = corr_tr_for_order.index[order_idx].tolist()

cmap = LinearSegmentedColormap.from_list('rho', ['#2166ac', '#f7f7f7', '#b2182b'], N=256)

for cohort_name, label in [('training', 'train'), ('validation', 'test')]:
    sub = df[df['cohort'] == cohort_name][protein_cols_all].apply(pd.to_numeric, errors='coerce')
    corr_c = sub[top_proteins].corr(method='spearman').loc[top_order, top_order]
    corr_c.to_csv(ch_dir / f'spearman_matrix_{label}.csv')

    n_c = int((df['cohort'] == cohort_name).sum())
    fig, ax = plt.subplots(figsize=(11, 10))
    im = ax.imshow(corr_c.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{label.title()} cohort (n={n_c})', fontsize=11)
    if label == 'test':
        plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label=r'Spearman $\rho$')

    out_path = PANEL_V2 / f'supp_fig_s2_corr_heatmap_{label}.svg'
    fig.savefig(out_path, dpi=600, bbox_inches='tight')

## Supplementary S3 — co-expression network sig prots
Pairwise Spearman correlation among the 17 pre-specified signature genes, edges shown for `|rho| >= 0.30` and BH-FDR < 0.05. Thick edges at `|rho| >= 0.50`

In [ ]:
sig_dir = ROOT / 'supp_signature_network'; sig_dir.mkdir(exist_ok=True)
CORE_SIGS = {k: SIGNATURE_DEFS[k] for k in ['IFN_Chemokine', 'Checkpoint_Cytokine', 'Innate_Inflammatory']}
SIG_COLORS = {'IFN_Chemokine': '#9467bd', 'Checkpoint_Cytokine': '#2ca02c', 'Innate_Inflammatory': '#ff7f0e'}
sig_genes = sorted({g for gs in SIGNATURE_DEFS.values() for g in gs if g in df.columns})
gene_mem = {g: [s for s, mem in CORE_SIGS.items() if g in mem] for g in sig_genes}

union_edges = set()
for cohort_name in ['training', 'validation']:
    x = df.loc[df['cohort'] == cohort_name, sig_genes].apply(pd.to_numeric, errors='coerce')
    for i, u in enumerate(sig_genes):
        for v in sig_genes[i + 1:]:
            rho, _ = stats.spearmanr(x[u], x[v])
            if np.isfinite(rho) and abs(rho) >= 0.30:
                union_edges.add((u, v))
G_union = nx.Graph(); G_union.add_nodes_from(sig_genes); G_union.add_edges_from(union_edges)
try:
    pos_shared = nx.kamada_kawai_layout(G_union, weight=None)
except Exception:
    pos_shared = nx.spring_layout(G_union, seed=42, k=1.5)

drawn_edge_sets = {}
for cohort_name, label in [('training', 'train'), ('validation', 'test')]:
    x = df.loc[df['cohort'] == cohort_name, sig_genes].apply(pd.to_numeric, errors='coerce')
    pairs = [(u, v, *stats.spearmanr(x[u], x[v])) for i, u in enumerate(sig_genes) for v in sig_genes[i + 1:]]
    sig_edges_c = pd.DataFrame(pairs, columns=['u', 'v', 'rho', 'p'])
    sig_edges_c['fdr'] = multipletests(sig_edges_c['p'], method='fdr_bh')[1]
    sig_edges_c.to_csv(sig_dir / f'edges_full_{label}.csv', index=False)
    draw = sig_edges_c[(sig_edges_c['fdr'] < 0.05) & (sig_edges_c['rho'].abs() >= 0.30)]
    drawn_edge_sets[label] = set(map(tuple, draw[['u', 'v']].values))  # edges kept in this cohort
    print(f'[{label}] n={len(x)}; edges drawn = {len(draw)} of {len(sig_edges_c)} '
          f'(thick |rho|>=0.50: {(draw["rho"].abs() >= 0.50).sum()})')
    G = nx.from_pandas_edgelist(draw, 'u', 'v', ['rho']); G.add_nodes_from(sig_genes)
    fig, ax = plt.subplots(figsize=(9, 8))
    draw_sig_network(ax, G, pos_shared, gene_mem, SIG_COLORS, n_cohort=len(x), n_drawn=len(draw), label=label)
    ax.legend(handles=[
        Line2D([0], [0], marker='o', color='w', markerfacecolor=SIG_COLORS['IFN_Chemokine'], markersize=10, label='IFN_Chemokine'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=SIG_COLORS['Checkpoint_Cytokine'], markersize=10, label='Checkpoint_Cytokine'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=SIG_COLORS['Innate_Inflammatory'], markersize=10, label='Innate_Inflammatory'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#bbbbbb', markersize=10, label='Integrated_Immune only'),
        Line2D([0], [0], color='#2166ac', lw=3, label=r'positive $\rho$'),
    ], loc='lower left', frameon=False, fontsize=8)
    fig.savefig(PANEL_V2 / f'supp_fig_s3_signature_network_{label}.svg', dpi=600, bbox_inches='tight')

_tr, _te = drawn_edge_sets['train'], drawn_edge_sets['test']
_shared = _tr & _te
_n_possible = len(sig_genes) * (len(sig_genes) - 1) // 2
_expected = len(_tr) * len(_te) / _n_possible
_jaccard = len(_shared) / len(_tr | _te) if (_tr or _te) else float('nan')
_hyp_p = stats.hypergeom.sf(len(_shared) - 1, _n_possible, len(_tr), len(_te))
print(f'[network overlap] shared edges = {len(_shared)} of {_n_possible} possible '
      f'(train {len(_tr)}, test {len(_te)}); Jaccard = {_jaccard:.3f}; '
      f'{len(_shared)}/{len(_tr)} = {len(_shared)/len(_tr):.0%} of training edges recovered in test; '
      f'expected by chance = {_expected:.1f}; hypergeometric P(overlap>={len(_shared)}) = {_hyp_p:.2e}')
pd.DataFrame(sorted(_shared), columns=['u', 'v']).to_csv(sig_dir / 'edges_shared_train_test.csv', index=False)

## Supplementary S4 — Pre-ranked GSEA on Inflamed vs Non-inflamed clusters

Proteins are ranked by `sign(log2FC) × -log10(p)` from the per-protein Welch t-test

- **Hallmark**
- **C7 ImmuneSigDB**
- **Reactome**

In [ ]:
gsea_dir = ROOT / 'supp_gsea'; gsea_dir.mkdir(exist_ok=True)
msig = Msigdb(); DBVER = '2024.1.Hs'
COLLECTIONS = [
    ('hallmark', 'h.all', 'Hallmark', 'HALLMARK_'),
    ('c7', 'c7.immunesigdb', 'C7 ImmuneSigDB', ''),
    ('reactome', 'c2.cp.reactome', 'Reactome', 'REACTOME_')]

def _pretty_term(term, prefix):
    t = term[len(prefix):] if prefix and term.startswith(prefix) else term
    return t.replace('_', ' ').title()

ranked_lists = {}
for cohort_label, p_col, lfc_col in [('train', 'train_p_value', 'train_log2FC'), ('test', 'test_p_value', 'test_log2FC')]:
    metric = -np.log10(np.clip(de[p_col].values, 1e-300, 1)) * np.sign(de[lfc_col].values)
    ranked_lists[cohort_label] = pd.DataFrame({'gene': de['protein'].values, 'metric': metric}).dropna() \
        .drop_duplicates('gene').sort_values('metric', ascending=False).reset_index(drop=True)
    ranked_lists[cohort_label].to_csv(gsea_dir / f'ranked_{cohort_label}.csv', index=False)

TOP_N = 15
for slug, category, display_name, prefix in COLLECTIONS:
    print(f'{display_name} ({category})')
    gene_sets = msig.get_gmt(category=category, dbver=DBVER)
    print(f'  {len(gene_sets)} gene sets fetched from MSigDB {DBVER}')
    for cohort_label in ['train', 'test']:
        pre_res = gp.prerank(rnk=ranked_lists[cohort_label], gene_sets=gene_sets, outdir=None,
                             min_size=5, max_size=500, permutation_num=1000, seed=42, no_plot=True, verbose=False)
        res_df = pre_res.res2d.copy()
        res_df['cohort'] = cohort_label; res_df['collection'] = slug
        res_df['pretty'] = res_df['Term'].apply(lambda t: _pretty_term(t, prefix))
        res_df = res_df.sort_values('NES', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
        res_df.to_csv(gsea_dir / f'gsea_{slug}_{cohort_label}.csv', index=False)
        print(f'  [{cohort_label}] {len(res_df)} pathways; {(res_df["FDR q-val"] < 0.05).sum()} FDR<0.05')
        plot_gsea_bar(res_df, slug=slug, cohort_label=cohort_label, display_name=display_name, top_n=TOP_N)

for slug, _, display_name, _ in COLLECTIONS:
    top_tr = set(pd.read_csv(gsea_dir / f'gsea_{slug}_train.csv').head(TOP_N)['Term'])
    top_te = set(pd.read_csv(gsea_dir / f'gsea_{slug}_test.csv').head(TOP_N)['Term'])
    _log(f'GSEA {display_name}: top-{TOP_N} pathways shared train↔test', f'{len(top_tr & top_te)}/{TOP_N}')


## Supplementary S5 — Immune-Apoptotic 7-protein composite KM

In [ ]:
apopt_sig = 'Immune_Apoptotic'
gcol = f'{apopt_sig}_group'

fig = plt.figure(figsize=(11, 11))
gs = fig.add_gridspec(2, 2, hspace=0.25, wspace=0.22)

for row, endpoint in enumerate(['OS', 'PFS']):
    tcol = 'estimated_OS_months' if endpoint == 'OS' else 'estimated_PFS_months'
    ecol = 'estimated_OS_event' if endpoint == 'OS' else 'estimated_PFS_event'
    xmax = 64 if endpoint == 'OS' else 52
    for col, (mask_, label, cohort) in enumerate([
        (is_train, 'Training', 'training'),
        (is_test,  'Test',     'validation'),
    ]):
        sub = df.loc[mask_]
        km_risk_pair(fig, gs, row, col, sub, tcol, ecol, gcol,
                     times=RISK_TIMES, xmax=xmax, title=f'{label} {endpoint}',
                     palette=KM_PALETTE_HL)
        _log(f'S6 KM log-rank P — Immune_Apoptotic {cohort} {endpoint}',
             f"{logrank_p(sub[tcol], sub[ecol], (sub[gcol]=='High').astype(int)):.3g}")

fig.suptitle('Immune-Apoptotic  Kaplan-Meier', fontsize=14, y=0.98)
svg = PANEL_V2 / 'supp_fig_s5_apoptotic_km.svg'
fig.savefig(svg, dpi=600, bbox_inches='tight')


## Supplementary S6 RF
 kept as model sensitivity check


In [ ]:
fig3_outdir = ROOT / 'figure3_response_classification'
fig3_outdir.mkdir(exist_ok=True)

rf_df = df.copy().reset_index(drop=True)
rf_df.to_csv(fig3_outdir / 'input_dataset_trimmed_last8_removed_notebook.csv', index=False)

_rf_exclude = {
    'patient_id','Provenance','Protocole','sexe','sex_completed','sex_source','age_years','cohort',
    'Karnofsky_status','Karnofsky_group','estimated_overall_risk_score','estimated_overall_risk_group',
    'estimated_OS_months','estimated_OS_event','estimated_PFS_months','estimated_PFS_event',
}
_sig_roots = list(SIGNATURE_DEFS)
_extra_meta = {'male', 'age_dec', 'age_ge60', 'kps_ge70', 'MSKCC_like',
               'bad_responder', 'long_survivor', 'outlier_score'}
rf_protein_cols = [
    c for c in rf_df.columns
    if c not in _rf_exclude
    and c not in _extra_meta
    and not c.startswith('z_')
    and c not in _sig_roots
    and not c.endswith('_group')
    and not any(c.startswith(r+'_') for r in _sig_roots)
]

_cohort_lc = rf_df['cohort'].astype(str).str.lower()
train_df = rf_df.loc[_cohort_lc.eq('training')].copy().reset_index(drop=True)
test_df  = rf_df.loc[_cohort_lc.eq('validation')].copy().reset_index(drop=True)

tasks = {
    'bad_pfs9': dict(
        title='Bad responder classifier',
        subtitle='Positive class: PFS event < 9 months',
        y_train=((train_df['estimated_PFS_event'].eq(1)) & (train_df['estimated_PFS_months'] < 9)).astype(int).to_numpy(),
        y_test =((test_df ['estimated_PFS_event'].eq(1)) & (test_df ['estimated_PFS_months'] < 9)).astype(int).to_numpy(),
        pos_label='Bad responder', neg_label='Other patients',
        prob_label='Probability of bad response',
        panel_letter_top='A', panel_letter_bottom='C',
        pos_color='#D55E00', neg_color='#56B4E9',
    ),
    'long_os12': dict(   # dict KEY left unchanged (referenced by the plot cells); endpoint is now OS>18mo
        title='Long survival classifier',
        subtitle='Positive class: OS > 18 months',
        # OS>18 = 1; dead by 18mo = 0; censored before 18mo = NaN (none in this cohort)
        y_train=np.where(train_df['estimated_OS_months'] > 18, 1.0,
                 np.where((train_df['estimated_OS_months'] <= 18) & (train_df['estimated_OS_event'] == 1), 0.0, np.nan)),
        y_test =np.where(test_df['estimated_OS_months'] > 18, 1.0,
                 np.where((test_df['estimated_OS_months'] <= 18) & (test_df['estimated_OS_event'] == 1), 0.0, np.nan)),
        pos_label='OS > 18 mo', neg_label='OS ≤ 18 mo (died)',
        prob_label='Probability of long survival',
        panel_letter_top='B', panel_letter_bottom='D',
        pos_color='#009E73', neg_color='#CC79A7',
    ),
}
Xtr_full = train_df[rf_protein_cols].copy()
Xte_full = test_df[rf_protein_cols].copy()

In [ ]:
def rank_features_ttest(X, y):
    rows = []
    for c in X.columns:
        a = pd.to_numeric(X.loc[y==1, c], errors='coerce')
        b = pd.to_numeric(X.loc[y==0, c], errors='coerce')
        stat, p = stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')
        mean1 = np.nanmean(a)
        mean0 = np.nanmean(b)
        rows.append((c, abs(0 if np.isnan(stat) else stat), 0 if np.isnan(mean1-mean0) else (mean1-mean0), 1 if np.isnan(p) else p))
    feat = pd.DataFrame(rows, columns=['protein','abs_t','mean_diff','p'])
    return feat.sort_values(['abs_t','p'], ascending=[False,True]).reset_index(drop=True)

In [ ]:
results = {}
for key, meta in tasks.items():
    # Exclude rows with a missing label (long-survival endpoint: patients censored before 18 months).
    ytr = np.asarray(meta['y_train'], dtype=float)
    yte = np.asarray(meta['y_test'],  dtype=float)
    _vtr = ~np.isnan(ytr); _vte = ~np.isnan(yte)
    ytr = ytr[_vtr].astype(int); yte = yte[_vte].astype(int)
    Xtr_t = Xtr_full.iloc[_vtr]; Xte_t = Xte_full.iloc[_vte]
    train_df_t = train_df.iloc[_vtr].reset_index(drop=True)
    test_df_t  = test_df.iloc[_vte].reset_index(drop=True)
    feat_rank = rank_features_ttest(Xtr_t, ytr)
    selected = feat_rank.head(40)['protein'].tolist()
    pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(
            n_estimators=800, criterion='gini', max_depth=None, min_samples_split=4,
            min_samples_leaf=2, max_features='sqrt', bootstrap=True,
            class_weight='balanced_subsample', random_state=42, n_jobs=1
        ))
    ])
    pipe.fit(Xtr_t[selected], ytr)
    ptr = pipe.predict_proba(Xtr_t[selected])[:, 1]
    pte = pipe.predict_proba(Xte_t[selected])[:, 1]
    auc_tr = roc_auc_score(ytr, ptr)
    auc_te = roc_auc_score(yte, pte)
    fpr_tr, tpr_tr, _ = roc_curve(ytr, ptr)
    fpr_te, tpr_te, _ = roc_curve(yte, pte)
    imp = pd.Series(pipe.named_steps['clf'].feature_importances_, index=selected).sort_values(ascending=False)
    top_prots = imp.head(5).index.tolist()
    box_frames = []
    for cohort_name, subdf, y, probs in [('Training', train_df_t, ytr, ptr), ('Test', test_df_t, yte, pte)]:
        tmp = subdf[top_prots].copy()
        tmp['cohort_plot'] = cohort_name
        tmp['class_binary'] = y
        tmp['class_label'] = np.where(y==1, meta['pos_label'], meta['neg_label'])
        tmp['display_group'] = np.where(
            (tmp['cohort_plot']=='Training') & (tmp['class_binary']==0), 'Train ' + meta['neg_label'],
            np.where((tmp['cohort_plot']=='Training') & (tmp['class_binary']==1), 'Train ' + meta['pos_label'],
                     np.where((tmp['cohort_plot']=='Test') & (tmp['class_binary']==0), 'Test ' + meta['neg_label'], 'Test ' + meta['pos_label']))
        )
        box_frames.append(tmp)
    results[key] = {
        'meta': meta, 'ptr': ptr, 'pte': pte, 'ytr': ytr, 'yte': yte,
        'auc_train': auc_tr, 'auc_test': auc_te,
        'fpr_train': fpr_tr, 'tpr_train': tpr_tr, 'fpr_test': fpr_te, 'tpr_test': tpr_te,
        'top_proteins': top_prots, 'box_df': pd.concat(box_frames, ignore_index=True)}

### panel helpers

In [ ]:
def add_count_bars(ax, y_train, y_test, meta):
    tr = pd.Series(y_train).value_counts().reindex([0,1], fill_value=0)
    te = pd.Series(y_test).value_counts().reindex([0,1], fill_value=0)
    x = np.array([0,1])
    width = 0.36
    ax.bar(x-width/2, [tr[0], tr[1]], width=width, color=[meta['neg_color'], meta['pos_color']], alpha=0.9, edgecolor='white')
    ax.bar(x+width/2, [te[0], te[1]], width=width, color=[meta['neg_color'], meta['pos_color']], alpha=0.45, edgecolor='white')
    for xi, yi in zip(x-width/2, [tr[0], tr[1]]):
        ax.text(xi, yi + 1.0, str(int(yi)), ha='center', va='bottom', fontsize=8)
    for xi, yi in zip(x+width/2, [te[0], te[1]]):
        ax.text(xi, yi + 1.0, str(int(yi)), ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([meta['neg_label'], meta['pos_label']], rotation=18, ha='right')
    ax.set_ylabel('Count')
    ax.set_title('Class counts', fontsize=10)
    ax.legend(handles=[Patch(facecolor='gray', alpha=0.9, label='Training'), Patch(facecolor='gray', alpha=0.45, label='Test')], frameon=False, fontsize=8, loc='upper right')
    sns.despine(ax=ax)

def add_roc(ax, res, meta):
    ax.plot(res['fpr_train'], res['tpr_train'], color=meta['pos_color'], lw=2.2, label=f"Training AUC = {res['auc_train']:.2f}")
    ax.plot(res['fpr_test'], res['tpr_test'], color=meta['neg_color'], lw=2.2, label=f"Test AUC = {res['auc_test']:.2f}")
    ax.plot([0,1],[0,1], ls='--', color='0.7', lw=1)
    ax.set_xlim(0,1)
    ax.set_ylim(0,1.02)
    ax.set_xlabel('1 - specificity')
    ax.set_ylabel('Sensitivity')
    ax.set_title(meta['subtitle'], fontsize=10)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
    sns.despine(ax=ax)

def add_density(ax, probs, y, meta, cohort_name, show_xlabel):
    neg = probs[np.asarray(y)==0]
    pos = probs[np.asarray(y)==1]
    if len(np.unique(neg)) > 1 and len(neg) > 1:
        sns.kdeplot(x=neg, ax=ax, fill=True, color=meta['neg_color'], alpha=0.45, linewidth=1)
    elif len(neg) > 0:
        ax.axvline(float(np.mean(neg)), color=meta['neg_color'], lw=2, alpha=0.8)
    if len(np.unique(pos)) > 1 and len(pos) > 1:
        sns.kdeplot(x=pos, ax=ax, fill=True, color=meta['pos_color'], alpha=0.45, linewidth=1)
    elif len(pos) > 0:
        ax.axvline(float(np.mean(pos)), color=meta['pos_color'], lw=2, alpha=0.8)
    ax.set_xlim(0,1)
    ax.set_ylabel('Density')
    ax.set_title(cohort_name, fontsize=9, pad=3)
    if show_xlabel:
        ax.set_xlabel(meta['prob_label'])
    else:
        ax.set_xlabel('')
        ax.set_xticklabels([])
    ax.legend(handles=[Patch(facecolor=meta['neg_color'], alpha=0.45, label=meta['neg_label']), Patch(facecolor=meta['pos_color'], alpha=0.45, label=meta['pos_label'])], frameon=False, fontsize=7, loc='upper right')
    sns.despine(ax=ax)

def add_boxplots(fig, subspec, box_df, top_proteins, meta):
    inner = gridspec.GridSpecFromSubplotSpec(len(top_proteins), 1, subplot_spec=subspec, hspace=0.14)
    group_order = ['Train ' + meta['neg_label'], 'Train ' + meta['pos_label'], 'Test ' + meta['neg_label'], 'Test ' + meta['pos_label']]
    palette = {group_order[0]: meta['neg_color'], group_order[1]: meta['pos_color'], group_order[2]: meta['neg_color'], group_order[3]: meta['pos_color']}
    axes = []
    for i, prot in enumerate(top_proteins):
        ax = fig.add_subplot(inner[i,0])
        tmp = box_df[['display_group', prot]].rename(columns={prot:'NPX'})
        sns.boxplot(data=tmp, x='display_group', y='NPX', order=group_order, palette=palette, width=0.62, fliersize=0, linewidth=1, ax=ax)
        sns.stripplot(data=tmp, x='display_group', y='NPX', order=group_order, palette=palette, alpha=0.55, size=1.8, ax=ax)
        ax.set_ylabel(prot, rotation=0, labelpad=28, fontsize=9, va='center')
        ax.yaxis.set_label_position('right')
        ax.yaxis.tick_right()
        if i < len(top_proteins)-1:
            ax.set_xlabel('')
            ax.set_xticklabels([])
        else:
            ax.set_xlabel('')
            ax.set_xticklabels(group_order, rotation=25, ha='right', fontsize=8)
        sns.despine(ax=ax)
        axes.append(ax)
    return axes

### plot

In [ ]:
def plot_fig3_counts_roc(task_key, letter, save_slug):
    res, meta = results[task_key], results[task_key]['meta']
    fig = plt.figure(figsize=(8.5, 4.4))
    gs  = fig.add_gridspec(1, 2, width_ratios=[0.6, 1.0], wspace=0.30)
    add_count_bars(fig.add_subplot(gs[0, 0]), res['ytr'], res['yte'], meta)
    add_roc       (fig.add_subplot(gs[0, 1]), res, meta)
    fig.suptitle(f'{meta["title"]}: class counts + ROC',
                 fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()
    return save_panel(fig, f'supp_fig_s6_rf_{letter}_{save_slug}')

def plot_fig3_density_box(task_key, letter, save_slug):
    res, meta = results[task_key], results[task_key]['meta']
    fig = plt.figure(figsize=(11, 4.8))
    gs  = fig.add_gridspec(1, 2, width_ratios=[0.9, 1.55], wspace=0.35)
    gs_dens = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[0, 0], hspace=0.30)
    add_density(fig.add_subplot(gs_dens[0, 0]), res['ptr'], res['ytr'], meta, 'Training', False)
    add_density(fig.add_subplot(gs_dens[1, 0]), res['pte'], res['yte'], meta, 'Test',     True)
    bx = add_boxplots(fig, gs[0, 1], res['box_df'], res['top_proteins'], meta)
    bx[0].text(0.5, 1.30, 'Top model proteins',
               transform=bx[0].transAxes, ha='center', fontsize=10)
    fig.suptitle(f'{meta["title"]}: density + top-protein NPX',
                 fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()
    return save_panel(fig, f'supp_fig_s6_rf_{letter}_{save_slug}')

In [ ]:
plot_fig3_counts_roc('bad_pfs9', 'A', 'counts_roc_bad_responder')

In [ ]:
plot_fig3_counts_roc('long_os12', 'B', 'counts_roc_long_survivor')

In [ ]:
plot_fig3_density_box('bad_pfs9', 'C', 'density_boxes_bad_responder')

In [ ]:
plot_fig3_density_box('long_os12', 'D', 'density_boxes_long_survivor')

In [ ]:
for k, lbl, tgt_train, tgt_test, tgt_pos_tr, tgt_pos_te in [
    ('bad_pfs9',  'Bad responder (PFS<9 mo)', 1.00, 0.70, 46, 40),
    ('long_os12', 'Long survival (OS>18 mo)', 1.00, 0.53, 84, 78),
]:
    r = results[k]
    pos_tr = int(r['ytr'].sum())
    pos_te = int(r['yte'].sum())
    _log(f'RF — {lbl} training AUC', f'{r["auc_train"]:.2f}')
    _log(f'RF — {lbl} test AUC',     f'{r["auc_test"]:.2f}')
    _log(f'RF — {lbl} positive counts (train/test)', f'{pos_tr}/{pos_te}')
    _log(f'RF — {lbl} top-5 proteins', r['top_proteins'])

## Supplementary S7 — Elastic-net signature enrichment

Hypergeometric test of the four immune signatures against the proteins


In [ ]:
conv_dir = ROOT / 'enet_signature_enrichment'; conv_dir.mkdir(exist_ok=True)
_M = len(protein_cols_all)
_PRIMARY_SEL = 'Elastic-net logistic (CV-tuned, nonzero)'
_ENET_GRID = {'clf__C': [0.05, 0.2, 1.0], 'clf__l1_ratio': [0.5, 0.9]}

def _hyper_against(selected, label, endpoint):
    sel = set(selected); N = len(sel); rows = []
    for term, genes in SIGNATURE_DEFS.items():
        gset = set(g for g in genes if g in protein_cols_all); nset = len(gset)
        x = len(sel & gset)
        p = stats.hypergeom.sf(x - 1, _M, nset, N) if (nset and N) else 1.0
        rows.append(dict(endpoint=endpoint, selection=label, set_size=N, term=term, sig_size=nset,
                         overlap=x, overlap_genes=','.join(sorted(sel & gset)), p_value=p))
    return rows

_conv_rows = []
_enet_selected = {}
for ecol, elabel in [('bad_responder', 'Bad responder (PFS<9mo)'),
                     ('long_survivor', 'Long survival (OS>18mo)')]:
    yfull = pd.to_numeric(df[ecol], errors='coerce')
    _vtr  = (is_train & yfull.notna()).values
    Xtr_e = df.loc[_vtr, protein_cols_all]; ytr_e = yfull[_vtr].astype(int).to_numpy()

    _pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                      ('clf', LogisticRegression(penalty='elasticnet', solver='saga',
                                                 class_weight='balanced', max_iter=5000, random_state=42))])
    _gs = GridSearchCV(_pipe, _ENET_GRID, scoring='roc_auc',
                       cv=StratifiedKFold(5, shuffle=True, random_state=1), n_jobs=-1).fit(Xtr_e, ytr_e)
    _coef = _gs.best_estimator_.named_steps['clf'].coef_.ravel()
    _enz = [p for p, c in zip(protein_cols_all, _coef) if c != 0]
    _enet_selected[elabel] = _enz
    _conv_rows += _hyper_against(_enz, _PRIMARY_SEL, elabel)
    _log(f'enet-enrich {elabel}',
         f"best C={_gs.best_params_['clf__C']} l1={_gs.best_params_['clf__l1_ratio']} "
         f"cvAUC={_gs.best_score_:.3f} n_selected={len(_enz)}")
    top40 = rank_features_ttest(Xtr_e, ytr_e).head(40)['protein'].tolist()
    _conv_rows += _hyper_against(top40, 'Top-40 |t| screen', elabel)

enet_enrichment = pd.DataFrame(_conv_rows)
enet_enrichment['adj_p'] = enet_enrichment.groupby(['endpoint', 'selection'])['p_value'].transform(
    lambda s: multipletests(s, method='fdr_bh')[1] if len(s) else s)
enet_enrichment.to_csv(conv_dir / 'enet_signature_enrichment.csv', index=False)
for _, r in enet_enrichment[enet_enrichment.selection.eq(_PRIMARY_SEL)].iterrows():
    _log(f'enet-enrich {r.endpoint} {r.term}', f'overlap={r.overlap}/{r.sig_size} p={r.p_value:.3g} FDR={r.adj_p:.3g}')
enet_enrichment[enet_enrichment.selection.eq(_PRIMARY_SEL)]

In [ ]:
_sub = enet_enrichment[enet_enrichment.selection.eq(_PRIMARY_SEL)].copy()
_eps = ['Bad responder (PFS<9mo)', 'Long survival (OS>18mo)']
_terms = list(SIGNATURE_DEFS)
figCv, axCv = plt.subplots(figsize=(6.6, 4.2))
sc = None
for xi, ep in enumerate(_eps):
    for yi, term in enumerate(_terms):
        r = _sub[(_sub.endpoint == ep) & (_sub.term == term)]
        if not len(r):
            continue
        r = r.iloc[0]
        sc = axCv.scatter(xi, yi, s=60 + 230 * r['overlap'],
                          c=[-np.log10(max(r['adj_p'], 1e-6))], cmap='viridis', vmin=0, vmax=3,
                          edgecolor='black', linewidth=0.6, zorder=3)
        if r['overlap'] > 0:
            axCv.text(xi, yi, str(int(r['overlap'])), ha='center', va='center', fontsize=8, color='white')
axCv.set_xticks(range(len(_eps))); axCv.set_xticklabels(['Bad responder\n(PFS<9mo)', 'Long survival\n(OS>18mo)'])
axCv.set_yticks(range(len(_terms))); axCv.set_yticklabels(_terms)
axCv.set_xlim(-0.5, len(_eps) - 0.5); axCv.set_ylim(-0.5, len(_terms) - 0.5)
axCv.set_title('Signature enrichment of elastic-net-selected proteins\n'
               '(CV-tuned penalty; number = overlapping proteins)', fontsize=10)
if sc is not None:
    cb = plt.colorbar(sc, ax=axCv, fraction=0.046, pad=0.04); cb.set_label('-log10 FDR')
figCv.tight_layout()
figCv.savefig(PANEL_V2 / 'supp_fig_s7_enet_enrichment.svg', bbox_inches='tight')
plt.show()

## Supplementary S8 — cohort drift

Classifier trying to tell training- from test-cohort


In [ ]:
adv_dir = ROOT / 'adversarial_validation'; adv_dir.mkdir(exist_ok=True)
_yadv = (df['cohort'].astype(str).str.lower() == 'training').astype(int).to_numpy()
_Xadv = df[protein_cols_all]
_cv = StratifiedKFold(5, shuffle=True, random_state=42)
_adv_logit = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                       ('clf', LogisticRegression(penalty='elasticnet', solver='saga', C=0.1, l1_ratio=0.5,
                                                  max_iter=5000, random_state=42))])
_adv_rf = Pipeline([('imp', SimpleImputer(strategy='median')),
                    ('clf', RandomForestClassifier(n_estimators=600, max_features='sqrt', random_state=42, n_jobs=-1))])

adv_rows = []
for _name, _mdl in [('Elastic-net logistic', _adv_logit), ('Random forest', _adv_rf)]:
    _s = cross_val_score(_mdl, _Xadv, _yadv, scoring='roc_auc', cv=_cv, n_jobs=-1)
    adv_rows.append(dict(model=_name, auc_mean=_s.mean(), auc_sd=_s.std()))
    _log(f'adversarial AUC ({_name})', f'{_s.mean():.3f} ± {_s.std():.3f}')
adversarial = pd.DataFrame(adv_rows)
adversarial.to_csv(adv_dir / 'adversarial_auc.csv', index=False)

_adv_logit.fit(_Xadv, _yadv)
_advcoef = pd.Series(_adv_logit.named_steps['clf'].coef_.ravel(), index=protein_cols_all)
_adv_nz = int((_advcoef != 0).sum())
_advcoef_top = _advcoef.reindex(_advcoef.abs().sort_values(ascending=False).index).head(12)
pd.DataFrame({'protein': _advcoef.index, 'coef': _advcoef.values}).to_csv(
    adv_dir / 'adversarial_discriminators.csv', index=False)
_log('adversarial elastic-net nonzero discriminators', _adv_nz)

_tc = _advcoef_top[::-1]
figAdv, axB = plt.subplots(figsize=(6.4, 4.6))
axB.barh(range(len(_tc)), _tc.values, color=['#c44e52' if v > 0 else '#4c78a8' for v in _tc.values],
         edgecolor='black', linewidth=0.5)
axB.set_yticks(range(len(_tc))); axB.set_yticklabels(_tc.index, fontsize=8)
axB.axvline(0, color='0.4', lw=1)
axB.set_xlabel('Elastic-net coefficient (+ = higher in training)')
figAdv.tight_layout()
figAdv.savefig(PANEL_V2 / 'supp_fig_s8_adversarial.svg', bbox_inches='tight')
plt.show()

## Supplementary S10 — Per-pathway survival

In [ ]:
pwc_dir = ROOT / 'supp_pathway_cox'; pwc_dir.mkdir(exist_ok=True)
hallmark = gp.get_library('MSigDB_Hallmark_2020')
hallmark_panel = {p: [g for g in genes if g in protein_cols_all]
                for p, genes in hallmark.items() if len([g for g in genes if g in protein_cols_all]) >= 5}
_log('Pathways with >=5 Olink panel members', len(hallmark_panel))

def _pathway_scores(cdf):
    out = {}
    for pathway, genes in hallmark_panel.items():
        x = cdf[genes].apply(pd.to_numeric, errors='coerce')
        z = ((x - x.mean()) / x.std(ddof=0)).fillna(0.0)
        s = z.mean(axis=1)
        out[pathway] = ((s - s.mean()) / s.std(ddof=0)).values
    return pd.DataFrame(out, index=cdf.index)

pw_train, pw_test = _pathway_scores(train_df), _pathway_scores(test_df)

def _cox_tmp(score, cdf, time_col, event_col):
    return pd.DataFrame({
        'time': pd.to_numeric(cdf[time_col], errors='coerce').values,
        'event': pd.to_numeric(cdf[event_col], errors='coerce').values,
        'score': score,
        'age_ge_60': (pd.to_numeric(cdf['age_years'], errors='coerce') >= 60).astype(int).values,
        'male': cdf['male'].values,
        'kps_ge_70': (pd.to_numeric(cdf['Karnofsky_status'], errors='coerce') >= 70).astype(int).values,
    }).dropna()

def _fit_pw_cox(tmp, adj=False):
    if len(tmp) < 10 or tmp['event'].sum() < 3:
        return (np.nan,) * 4
    cols = ['score'] + (['age_ge_60', 'male', 'kps_ge_70'] if adj else [])
    f = fit_cox_hr(tmp['time'], tmp['event'], tmp[cols].values, var_names=cols)
    return float(f['hr'][0]), float(f['lo'][0]), float(f['hi'][0]), float(f['p'][0])

cox_rows = []
for pathway, genes in hallmark_panel.items():
    n_g = len(genes)
    for cohort_label, cdf, pw in [('train', train_df, pw_train), ('test', test_df, pw_test)]:
        score = pw[pathway].values
        for endpoint, tc, ec in ENDPOINTS:
            tmp = _cox_tmp(score, cdf, tc, ec)
            for model, adj in [('univariate', False), ('adjusted', True)]:
                hr, lo, hi, p = _fit_pw_cox(tmp, adj)
                cox_rows.append((pathway, cohort_label, endpoint, n_g, model, hr, lo, hi, p))

cox_pw = pd.DataFrame(cox_rows, columns=['pathway','cohort','endpoint','n_genes','model','HR','HR_lo','HR_hi','p'])
cox_pw['fdr'] = np.nan
for _, sub in cox_pw.groupby(['cohort', 'endpoint', 'model']):
    m = sub['p'].notna()
    if m.any():
        cox_pw.loc[sub.index[m], 'fdr'] = multipletests(sub.loc[m, 'p'], method='fdr_bh')[1]
cox_pw['pretty'] = cox_pw['pathway'].str.replace('HALLMARK_', '').str.replace('_', ' ').str.title()
cox_pw.to_csv(pwc_dir / 'pathway_cox_results.csv', index=False)

top_pw = (cox_pw[(cox_pw.cohort == 'train') & (cox_pw.endpoint == 'OS') & (cox_pw.model == 'univariate')]
          .sort_values('fdr').head(10)['pathway'].tolist())
COLOR_TRAIN, COLOR_TEST = '#59a14f', '#9c6ade'
fig, axes = plt.subplots(2, 2, figsize=(13, 9.5), sharey=True)
for cohort_label, endpoint, ax, ttl, col in [
        ('train','OS', axes[0,0], 'Training — OS', COLOR_TRAIN),
        ('test', 'OS', axes[0,1], 'Test — OS', COLOR_TEST),
        ('train','PFS', axes[1,0], 'Training — PFS', COLOR_TRAIN),
        ('test', 'PFS', axes[1,1], 'Test — PFS', COLOR_TEST)]:
    sub = cox_pw[(cox_pw.cohort == cohort_label) & (cox_pw.endpoint == endpoint)]
    sub_u = sub[sub.model == 'univariate'].set_index('pathway').loc[top_pw].reset_index()
    sub_a = sub[sub.model == 'adjusted'].set_index('pathway').loc[top_pw].reset_index()
    draw_pathway_forest(ax, sub_u.to_dict('records'), sub_a.to_dict('records'),
                        sub_u['pretty'].tolist(), color=col)
    ax.set_xscale('log'); ax.set_xlim(0.5, 4)
    ax.xaxis.set_major_locator(FixedLocator([0.5, 0.75, 1, 1.5, 2, 3]))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.set_title(ttl, fontsize=11); ax.set_xlabel('Hazard ratio (per +1 SD pathway score)')

handles = [
    Line2D([0], [0], marker='o', color=COLOR_TRAIN, lw=1.8, markeredgecolor='black', markeredgewidth=0.35, label='Training, univariate'),
    Line2D([0], [0], marker='o', color=COLOR_TRAIN, lw=1.8, markerfacecolor='white', markeredgecolor=COLOR_TRAIN, markeredgewidth=1.5, label='Training, adjusted'),
    Line2D([0], [0], marker='o', color=COLOR_TEST, lw=1.8, markeredgecolor='black', markeredgewidth=0.35, label='Test, univariate'),
    Line2D([0], [0], marker='o', color=COLOR_TEST, lw=1.8, markerfacecolor='white', markeredgecolor=COLOR_TEST, markeredgewidth=1.5, label='Test, adjusted'),
]
fig.legend(handles=handles, loc='upper center', frameon=False, ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0.99))
fig.suptitle('Hallmark pathway scores as continuous Cox predictors (by training-OS univariate FDR)', fontsize=11, y=0.95)
fig.tight_layout(rect=[0, 0, 1, 0.91])
fig.savefig(PANEL_V2 / 'supp_fig_s10_pathway_cox_forest.svg', dpi=600, bbox_inches='tight')

for ep, mdl in [('OS','univariate'),('PFS','univariate'),('OS','adjusted'),('PFS','adjusted')]:
    n_tr = int((cox_pw[(cox_pw.cohort=='train') & (cox_pw.endpoint==ep) & (cox_pw.model==mdl)]['fdr'] < 0.05).sum())
    n_te = int((cox_pw[(cox_pw.cohort=='test') & (cox_pw.endpoint==ep) & (cox_pw.model==mdl)]['fdr'] < 0.05).sum())
    _log(f'Pathway Cox FDR<0.05  {mdl:11s} {ep}: train / test', f'{n_tr} / {n_te}')


### S10b multivariable Cox forests

In [ ]:
pmv_dir = ROOT / 'supp_pathway_multivar'; pmv_dir.mkdir(exist_ok=True)
top_3_pw = (cox_pw[(cox_pw.cohort=='train') & (cox_pw.endpoint=='PFS') & (cox_pw.model=='univariate')]
            .sort_values('p').head(3)['pathway'].tolist())
VAR_ORDER = ['score', 'age_ge_60', 'male', 'kps_ge_70']
VAR_LABELS = {'score': 'pathway score (+1 SD)', 'age_ge_60': 'age ≥ 60',
              'male': 'male sex', 'kps_ge_70': 'KPS ≥ 70'}
COLOR_TRAIN, COLOR_TEST = '#59a14f', '#9c6ade'

def _fit_pathway_mv(score, cdf, tc, ec):
    tmp = _cox_tmp(score, cdf, tc, ec)
    if len(tmp) < 10 or tmp['event'].sum() < 3:
        return None
    f = fit_cox_hr(tmp['time'], tmp['event'], tmp[VAR_ORDER].values, var_names=VAR_ORDER)
    if f['model'] is None:
        return None
    return {v: (float(f['hr'][i]), float(f['lo'][i]), float(f['hi'][i]), float(f['p'][i]))
            for i, v in enumerate(VAR_ORDER)}

fig, axes = plt.subplots(3, 2, figsize=(11.5, 9), sharey=True)
mv_rows = []
for ri, pathway in enumerate(top_3_pw):
    pretty_p = pathway.replace('HALLMARK_', '').replace('_', ' ').title()
    for ci, (endpoint, tc, ec) in enumerate(ENDPOINTS):
        ax = axes[ri, ci]; ax.axvline(1, color='black', ls='--', lw=1.0, zorder=1)
        y = np.arange(len(VAR_ORDER))[::-1]
        for cohort_label, cdf, pw, col, off in [
                ('train', train_df, pw_train, COLOR_TRAIN, +0.18),
                ('test', test_df, pw_test, COLOR_TEST, -0.18)]:
            fit = _fit_pathway_mv(pw[pathway].values, cdf, tc, ec)
            if fit is None: continue
            for k, v in enumerate(VAR_ORDER):
                hr, lo, hi, p = fit[v]
                yi = y[k] + off
                star = '*' if (pd.notna(p) and p < 0.05) else ''
                draw_hr_point(ax, hr, lo, hi, yi, color=col, filled=True, star=star, xtext=1.04)
                mv_rows.append({'pathway': pathway, 'cohort': cohort_label, 'endpoint': endpoint,
                                'covariate': v, 'HR': hr, 'HR_lo': lo, 'HR_hi': hi, 'p': p})
        ax.set_xscale('log'); ax.set_xlim(0.2, 12)
        ax.set_xticks([0.25, 0.5, 1, 2, 5, 10]); ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_yticks(y); ax.set_yticklabels([VAR_LABELS[v] for v in VAR_ORDER], fontsize=9)
        ax.set_title(f'{pretty_p} — {endpoint}', fontsize=10); ax.set_xlabel('Hazard ratio')

fig.legend(handles=[
    Line2D([0], [0], marker='o', color=COLOR_TRAIN, lw=1.8, markeredgecolor='black', markeredgewidth=0.35, label='Training'),
    Line2D([0], [0], marker='o', color=COLOR_TEST, lw=1.8, markeredgecolor='black', markeredgewidth=0.35, label='Test'),
], loc='upper center', frameon=False, ncol=2, fontsize=10, bbox_to_anchor=(0.5, 0.99))
fig.suptitle('Multivariable Cox forests for top-3 pathways by training-PFS p', fontsize=11, y=0.96)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(PANEL_V2 / 'supp_fig_s10_pathway_multivar_forest.svg', dpi=600, bbox_inches='tight')
pd.DataFrame(mv_rows).to_csv(pmv_dir / 'pathway_multivar_results.csv', index=False)
_log('Pathway multivariable Cox: 3 pathways x 2 endpoints x 2 cohorts =', f'{int(3*2*2)} models, 4 covariates each')


###  S10c KM

In [ ]:
pkm_dir = ROOT / 'supp_pathway_km'; pkm_dir.mkdir(exist_ok=True)
COLOR_HIGH, COLOR_LOW = '#c44e52', '#4c78a8'
fig, axes = plt.subplots(3, 4, figsize=(16, 9.5), sharey='row')
km_rows = []
panels = [
    ('train', train_df, pw_train, 'OS',  'estimated_OS_months',  'estimated_OS_event'),
    ('test',  test_df,  pw_test,  'OS',  'estimated_OS_months',  'estimated_OS_event'),
    ('train', train_df, pw_train, 'PFS', 'estimated_PFS_months', 'estimated_PFS_event'),
    ('test',  test_df,  pw_test,  'PFS', 'estimated_PFS_months', 'estimated_PFS_event'),
]
for ri, pathway in enumerate(top_3_pw):
    pretty_p = pathway.replace('HALLMARK_', '').replace('_', ' ').title()
    cut = float(np.median(pw_train[pathway].values))
    for ci, (cohort_label, cdf, pw, endpoint, tc, ec) in enumerate(panels):
        ax = axes[ri, ci]
        tmp = cdf.assign(_pw_score=pw[pathway].values, _pw_grp=np.where(pw[pathway].values > cut, 'High', 'Low'))
        tmp = tmp.dropna(subset=[tc, ec, '_pw_score'])
        n_h = int((tmp['_pw_grp'] == 'High').sum()); n_l = int((tmp['_pw_grp'] == 'Low').sum())
        plot_km(ax, tmp, tc, ec, '_pw_grp',
                palette={'High': COLOR_HIGH, 'Low': COLOR_LOW}, group_order=['High', 'Low'],
                xmax=max(float(pd.to_numeric(tmp[tc], errors='coerce').max() or 24), 24) * 1.02,
                title=f'{cohort_label.title()} — {endpoint}', legend_loc='lower left',
                legend_fontsize=7, censor_ticks=False, p_box=True, p_groups=('High', 'Low'),
                p_style='box', p_loc=(0.97, 0.05), p_ha='right',
                p_text=lambda p: f'log-rank P = {p:.3g}')
        p_lr = logrank_p(tmp[tc], tmp[ec], (tmp['_pw_grp'] == 'High').astype(int))
        km_rows.append({'pathway': pathway, 'cohort': cohort_label, 'endpoint': endpoint,
                        'train_median_cut': cut, 'n_high': n_h, 'n_low': n_l, 'logrank_p': float(p_lr)})
        if ci == 0: ax.set_ylabel(f'{pretty_p}\nSurvival probability', fontsize=9)
        if ri == 2: ax.set_xlabel(f'{endpoint} (months)')

fig.suptitle('Kaplan-Meier survival by top-3 Hallmark pathway scores', fontsize=11, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(PANEL_V2 / 'supp_fig_s10_pathway_km.svg', dpi=600, bbox_inches='tight')
pd.DataFrame(km_rows).to_csv(pkm_dir / 'pathway_km_results.csv', index=False)
for r in km_rows:
    _log(f"KM log-rank P — {r['pathway'].replace('HALLMARK_','')} {r['cohort']} {r['endpoint']}", f"{r['logrank_p']:.3g}")
_log('Pathway KM: 3 pathways x 2 endpoints x 2 cohorts =', f'{int(3*2*2)} log-rank tests')


## Summary

All metrics are written to `outputs/notebook_summary.csv`.


In [ ]:
summary = pd.DataFrame([{'metric': k, 'value': v} for k, v in _METRICS.items()])
summary_csv = Path('outputs/notebook_summary.csv')
summary_csv.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_csv, index=False)
print(f'Wrote {len(summary)} rows to {summary_csv}')
summary